# 🌱 GreenLedger — Campus Resource Optimisation (IC26-05-01)
### Reference campus: **Concordia University, Montréal** — real open data, nothing simulated

**In one line:** GreenLedger reads every campus meter, waste invoice and a live room camera, **recommends** what to change,
**controls** room AC/fan/lights from real occupancy, and **proves** with a ghost twin + placebo test whether a change really saved anything —
then reports the carbon honestly.

| Tab | What it does |
|---|---|
| 📊 Overview | this month vs the ghost twin · 🎚️ what-if sliders · top recommendations · map · report download |
| ✅ Act & Prove | trust test first · approve/reject recommendations · ghost twin + placebo-in-time verification |
| 🎥 Live room | camera people-count → AC / fan / lights decision (ASHRAE 55 + 90.1) |
| 📂 Data | alerts · carbon report (Scopes 1-2-3) · waste · e-waste (your real asset register) |

**How to run (Google Colab):**
1. Click the 📁 folder icon and upload `DATA_OUTPUT.csv`, `WASTE_INVOICES.csv`, `WASTE_TYPE.csv`, `BUILDING_LIST.csv`
   (optional: your `ewaste_register.csv`). If you skip this, cell 2 downloads the four files from Concordia's open-data site.
2. Run the cells **one at a time, top to bottom** (Shift + Enter). Each prints a short, plain result.
3. The last cell opens the app — click the **…gradio.live** link (allow camera access for the Live room tab).

Total run time ≈ 3 minutes.

## Cell 1 · Install
Installs the libraries: app (Gradio), charts (Plotly), holiday calendar, statistics, gradient boosting (LightGBM),
people detection (Ultralytics YOLOv8) and thermal comfort (pythermalcomfort, ASHRAE 55).

In [ ]:
!pip -q install gradio plotly holidays scipy lightgbm ultralytics pythermalcomfort


## Cell 2 · Find the data files
Finds your uploaded CSV files (even if Colab renamed them) — or downloads them from Concordia's official open-data site.

🎤 **Tell the judges:** *"Everything comes from Concordia University's public open data — real meters and real waste invoices. We didn't simulate or fill in a single number."*

In [ ]:
import os, re, csv, glob, io, json, zipfile, datetime, warnings, urllib.request
import numpy as np
import pandas as pd
from scipy import stats
import holidays
import lightgbm as lgb
import plotly.graph_objects as go
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

SOURCE_URL = "https://opendata.concordia.ca/datasets/Facilities/"
FILES = ["DATA_OUTPUT.csv", "WASTE_INVOICES.csv", "WASTE_TYPE.csv", "BUILDING_LIST.csv"]
OPTIONAL = ["POINT_LIST.csv"]          # Concordia's official sensor list, used if available
OUT_DIR = os.environ.get("GL_OUT_DIR", "/content/greenledger_out" if os.path.isdir("/content") else "./greenledger_out")
DATA_DIR = os.environ.get("GL_DATA_DIR", "/content" if os.path.isdir("/content") else ".")
os.makedirs(OUT_DIR, exist_ok=True)


def find(name):
    """Find a file even if it was renamed on upload (e.g. 'DATA_OUTPUT (1).csv' or 'abc-DATA_OUTPUT.csv')."""
    stem = name[:-4].upper()
    for root in [DATA_DIR, ".", "/content", "/content/drive/MyDrive"]:
        if not root or not os.path.isdir(root):
            continue
        cands = glob.glob(os.path.join(root, "*.csv")) + glob.glob(os.path.join(root, "*", "*.csv"))
        hits = []
        for p in cands:
            s_ = re.sub(r"\(\d+\)$", "", os.path.splitext(os.path.basename(p))[0].upper().replace(" ", ""))
            if (s_ == stem or s_.endswith("-" + stem)) and os.path.getsize(p) > 0:
                hits.append(p)
        if hits:
            return sorted(hits, key=len)[0]
    return None


PATHS = {f: find(f) for f in FILES + OPTIONAL}
missing = [f for f in FILES if not PATHS[f]]
for f in missing + [o for o in OPTIONAL if not PATHS[o]]:   # try Concordia's official open-data site
    if True:
        try:
            dst = os.path.join(OUT_DIR, f)
            urllib.request.urlretrieve(SOURCE_URL + f, dst)
            PATHS[f] = dst
            print(f"Downloaded {f} from {SOURCE_URL}")
        except Exception as e:
            print(f"Could not download {f}: {e}")
missing = [f for f in FILES if not PATHS[f]]
assert not missing, (f"Missing {missing}. In Colab: click the folder icon on the left, upload the 4 CSV files, "
                     f"then run this cell again.")
for f, p in PATHS.items():
    print(f"{f:20s} → {p or 'not found (optional)'}")
print("\nData: Concordia University open data (Facilities), licence CC BY 4.0 — "
      "https://www.concordia.ca/web/open-data.html")


## Cell 3 · Read the meters
Concordia's building-automation export is many reports stacked in one file. This cell stitches them into one hourly table and names each sensor from Concordia's own API documentation.

| Meter | What it measures |
|---|---|
| ⚡ Loyola campus | electricity demand of the whole Loyola campus (kW) |
| ⚡ EV building | electricity demand of the Engineering & Visual Arts building |
| ⚡ EV + GM + MB + TD | electricity of four downtown buildings together |
| 🔥 EV + GM + MB / GM | natural gas (running counters that reset each month) |
| 💧 Municipal water | the city water supply meter (running counter) |
| 🌡️ Weather | outdoor temperature from the EV rooftop station |

In [ ]:
# ---- What each sensor is: Concordia's own API documentation (opendataConcordiaU/documentation,
#      facilities/consumption and facilities/environmental) ----
METER_INFO = {
    # point:  (system name, resource, plain name, buildings it covers, reading type, unit)
    "Point_12": ("DEMANDE.ELEC.MOYEN60S.BS", "electricity", "Loyola campus",            "LOY campus", "power", "kW"),
    "Point_7":  ("TOTKWEV.IC",               "electricity", "EV building",              "EV",          "power", "kW"),
    "Point_6":  ("TOTKWEV-MB.IC",            "electricity", "EV + GM + MB + TD buildings", "EV,GM,MB,TD", "power", "kW"),
    "Point_8":  ("MDI.001-650.IC",           "gas",         "EV + GM + MB buildings",   "EV,GM,MB",    "counter", "m³ (as documented)"),
    "Point_9":  ("MDICOR.GM",                "gas",         "GM building",              "GM",          "counter", "m³ (as documented)"),
    "Point_10": ("MDE.EFDP.IC",              "water",       "Municipal water supply",   "",            "counter", "m³"),
}
# campus totals without double counting: EV is inside EV+GM+MB+TD, GM is inside EV+GM+MB
CAMPUS_TOTAL = {"electricity": ["Point_12", "Point_6"], "gas": ["Point_8"], "water": ["Point_10"]}
WEATHER_POINT = "Point_3"     # MTX1-017.IC — outdoor temperature, EV-building weather station (°C)
RESOURCE = {
    "electricity": {"label": "Electricity", "unit": "kWh", "color": "#10B981", "icon": "⚡"},
    "gas":         {"label": "Natural gas", "unit": "m³",  "color": "#F97316", "icon": "🔥"},
    "water":       {"label": "Water",       "unit": "m³",  "color": "#3B82F6", "icon": "💧"},
    "waste":       {"label": "Waste",       "unit": "t",   "color": "#F59E0B", "icon": "🗑️"},
}

# ---- read Concordia's building-automation export (it is many reports stacked in one file) ----
rows, header = [], None
with open(PATHS["DATA_OUTPUT.csv"], errors="replace") as fh:
    for r in csv.reader(fh):
        if r and r[0] == "<>Date":
            header = ["date", "time"] + r[2:]
            continue
        if header and len(r) >= len(header) and "/" in r[0]:
            rows.append(r[:len(header)])
raw = pd.DataFrame(rows, columns=header)
stamp = pd.to_datetime(raw["date"] + " " + raw["time"].str.strip(), format="%m/%d/%Y %H:%M:%S")
raw = raw.drop(columns=["date", "time"]).apply(lambda s: pd.to_numeric(s.str.strip(), errors="coerce"))  # "No Data" → missing
raw.index = stamp.dt.floor("h").values
raw = raw[~raw.index.duplicated(keep="last")].sort_index()           # overlapping reports: keep the latest
HOURLY = raw.reindex(pd.date_range(raw.index.min(), raw.index.max(), freq="h"))
DAYS = pd.date_range(HOURLY.index.min().normalize(), HOURLY.index.max().normalize(), freq="D")

# optional: Concordia's official point list (adds building / description if the file is available)
if PATHS.get("POINT_LIST.csv"):
    try:
        _pl = pd.read_csv(PATHS["POINT_LIST.csv"])
        print("Official point list loaded:", list(_pl.columns))
    except Exception as e:
        print("POINT_LIST.csv could not be read:", e)

METERS = pd.DataFrame([{"point": p, "system name": v[0], "resource": v[1], "meter": v[2], "buildings": v[3] or "not stated",
                        "reading": "average power (kW)" if v[4] == "power" else "running total (resets each month/day)",
                        "first reading": HOURLY[p].first_valid_index(), "last reading": HOURLY[p].last_valid_index(),
                        "hours with data": f"{HOURLY[p].notna().mean():.0%}"} for p, v in METER_INFO.items()])
print(f"Hourly readings: {HOURLY.index.min():%d %b %Y} → {HOURLY.index.max():%d %b %Y}  ({len(HOURLY):,} hours)")
print(METERS.drop(columns=["point"]).to_string(index=False))


## Cell 4 · Clean the data (3 simple rules)
1. **Incomplete day** — fewer than 22 of 24 hours recorded → not used.
2. **Impossible jump** — one hour can't use more than the meter ever used in a whole day → that hour is removed.
3. **Not measuring** — the counter rose by the same amount every hour all day → it was estimating, not measuring.

🎤 *"We found real faults — the GM gas meter jumped 306,266 units in one hour; the water meter flat-lined for 5 weeks. We show every excluded day and why. We never fill gaps."*

In [ ]:
# ---- Turn hourly readings into one number per meter per day, with three plain quality rules ----
COMPLETE = 0.90      # a day counts only if ≥90% of its hours have a reading (22 of 24)
FAULTS = []          # every excluded day, with the reason — shown in the app


def daily_from_power(p):
    s = HOURLY[p]
    n = s.resample("D").count()
    return (s.resample("D").mean() * 24).where(n >= COMPLETE * 24).reindex(DAYS)   # average kW × 24 h = kWh


def hourly_use_from_counter(p):
    s = HOURLY[p]
    use = s.diff()
    reset = use < 0
    use[reset] = s[reset]                       # counter restarted at zero → the reading is the use since restart
    use = use.where(s.notna() & s.shift(1).notna())
    return use, reset


def daily_from_counter(p):
    use, reset = hourly_use_from_counter(p)
    # Rule A — impossible jump: one hour cannot use more than the most this meter ever used in a whole day
    while True:
        tot = use.resample("D").sum(min_count=1).dropna().sort_values(ascending=False)
        limit = np.where(use.index.normalize() == tot.index[0], tot.iloc[1], tot.iloc[0])
        bad = use > limit
        if not bad.any():
            break
        for t in use[bad].index:
            FAULTS.append({"point": p, "day": t.normalize(), "rule": "impossible jump",
                           "detail": f"{use[t]:,.0f} units in one hour at {t:%H:%M} — more than any whole day"})
        use[bad] = np.nan
    # Rule B — not measuring: every hourly step identical (±1 unit) all day = a fixed estimate, not a reading
    u = use[~reset]
    g = u.groupby(u.index.normalize())
    stuck = ((g.max() - g.min()) <= 1) & (g.count() >= COMPLETE * 24 - 1)
    for d in stuck[stuck].index:
        FAULTS.append({"point": p, "day": d, "rule": "not measuring",
                       "detail": f"same amount every hour ({g.median()[d]:,.0f}/h) — meter was not measuring"})
    n = use.resample("D").count()
    day = (use.resample("D").mean() * 24).where(n >= COMPLETE * 24).reindex(DAYS)
    day[day.index.isin(stuck[stuck].index)] = np.nan
    return day


DAILY = pd.DataFrame({p: (daily_from_power(p) if v[4] == "power" else daily_from_counter(p))
                      for p, v in METER_INFO.items()})
for p in DAILY:
    miss = DAILY[p].isna() & HOURLY[p].resample("D").count().reindex(DAYS).gt(0)
    for d in DAILY.index[miss]:
        if not any(f["point"] == p and f["day"] == d for f in FAULTS):
            FAULTS.append({"point": p, "day": d, "rule": "incomplete day", "detail": "fewer than 22 of 24 hours recorded"})
FAULTS = pd.DataFrame(FAULTS, columns=["point", "day", "rule", "detail"]).sort_values(["point", "day"])
FAULTS["meter"] = FAULTS.point.map(lambda p: f"{RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}")


def fault_periods(F):
    """Consecutive days with the same meter and rule → one line (easier to read)."""
    out = []
    for (m, rule), g in F.sort_values("day").groupby(["meter", "rule"]):
        days = list(g.day); det = list(g.detail); i = 0
        while i < len(days):
            j = i
            while j + 1 < len(days) and days[j + 1] - days[j] == pd.Timedelta(days=1):
                j += 1
            out.append({"meter": m, "from": days[i].date(), "to": days[j].date(), "days": j - i + 1, "rule": rule,
                        "detail": det[i] if i == j else det[i].split(" — ")[-1]})
            i = j + 1
    return pd.DataFrame(out).sort_values("from")

# daily outdoor temperature (EV weather station) + Quebec public holidays (official calendar via `holidays`)
TEMP = HOURLY[WEATHER_POINT].resample("D").mean().where(HOURLY[WEATHER_POINT].resample("D").count() >= COMPLETE * 24).reindex(DAYS)
QC_HOLIDAYS = holidays.Canada(subdiv="QC", years=sorted(set(DAYS.year)))
OFF_DAY = pd.Series([(d.dayofweek >= 5) or (d.date() in QC_HOLIDAYS) for d in DAYS], index=DAYS)

print("Good days per meter (out of", len(DAYS), "):")
for p in DAILY:
    print(f"  {RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]:30s} {DAILY[p].notna().sum():4d} good days")
print("\nExcluded days by reason:")
print(FAULTS.groupby(["meter", "rule"]).size().rename("days").to_string())
print("\nThe problems (not counting incomplete days):")
print(fault_periods(FAULTS[FAULTS.rule != "incomplete day"]).to_string(index=False))


## Cell 5 · Learn "normal" — the ghost twin's brain
Two models are trained on one full year (every season once), using the day's **temperature**, **weekday/holiday** and **season**:
* the **ASHRAE change-point model** (explainable: "+1,233 kWh per °C colder"), and
* **gradient boosting (LightGBM)** — machine learning.

Each baseline month is hidden in turn and predicted by both; **whichever misses less becomes that meter's ghost twin.**
The change-point model is always kept to explain the building and to grade accuracy (ASHRAE Guideline 14).

🎤 *"We don't pick AI or physics by taste — each meter gets the model that predicts months it has never seen better."*

In [ ]:
# ---- "Expected use": how much each meter normally uses for a given temperature and day type ----
# ASHRAE change-point model (ASHRAE Guideline 14 / Inverse Modeling Toolkit, RP-1050):
#   daily use = base + (off-day change) + heating slope × max(0, Th − temp) + cooling slope × max(0, temp − Tc)
# The balance temperatures Th, Tc are found from the data (every whole °C observed is tried); the simplest
# model that fits is chosen by BIC. Slopes are kept ≥ 0 (colder can't reduce heating; hotter can't reduce cooling).
_tm = TEMP.notna().groupby(TEMP.index.to_period("M")).mean()
BASE_START = _tm[_tm >= COMPLETE].index[0].to_timestamp()               # first month with weather data
BASE_END = BASE_START + pd.DateOffset(years=1) - pd.Timedelta(days=1)    # one full year = every season once
MON_START = BASE_END + pd.Timedelta(days=1)
print(f"Baseline year: {BASE_START:%d %b %Y} → {BASE_END:%d %b %Y}   ·   Monitoring: {MON_START:%d %b %Y} → {DAYS[-1]:%d %b %Y}")


def _X(idx, th, tc):
    t = TEMP.reindex(idx).values
    cols = [np.ones(len(idx)), OFF_DAY.reindex(idx).values.astype(float)]
    if th is not None: cols.append(np.maximum(0, th - t))
    if tc is not None: cols.append(np.maximum(0, t - tc))
    return np.column_stack(cols)


def _ml_features(idx):
    idx = pd.DatetimeIndex(idx)
    return pd.DataFrame({"temp": TEMP.reindex(idx).values, "off": OFF_DAY.reindex(idx).values.astype(int),
                         "dow": idx.dayofweek, "doy_sin": np.sin(2 * np.pi * idx.dayofyear / 365.25),
                         "doy_cos": np.cos(2 * np.pi * idx.dayofyear / 365.25)}, index=idx)


def fit_model(y, kind="cp"):
    """kind='cp': ASHRAE change-point model · kind='ml': gradient boosting (LightGBM, library defaults)."""
    if kind == "ml":
        y = y.dropna(); y = y[TEMP.reindex(y.index).notna()]
        mdl = lgb.LGBMRegressor(random_state=0, verbosity=-1).fit(_ml_features(y.index), y.values)
        return {"kind": "ml", "mdl": mdl, "idx": y.index, "n": len(y)}
    y = y.dropna()
    y = y[TEMP.reindex(y.index).notna()]
    t = TEMP.reindex(y.index).values
    o = OFF_DAY.reindex(y.index).values.astype(float)
    v = y.values
    n = len(v)
    grid = [None] + sorted(set(np.round(t).astype(int)))
    best = None
    for th in grid:
        h = None if th is None else np.maximum(0, th - t)
        for tc in grid:
            if th is not None and tc is not None and tc < th:
                continue
            cols = [np.ones(n), o] + ([h] if h is not None else []) + ([np.maximum(0, t - tc)] if tc is not None else [])
            X = np.column_stack(cols)
            b = np.linalg.lstsq(X, v, rcond=None)[0]
            if len(b) > 2 and b[2:].min() < 0:
                continue
            sse = float(((X @ b - v) ** 2).sum())
            k = X.shape[1] + (th is not None) + (tc is not None)
            bic = n * np.log(sse / n) + k * np.log(n)
            if best is None or bic < best["bic"]:
                best = {"kind": "cp", "bic": bic, "th": th, "tc": tc, "b": b, "k": k, "n": n, "idx": y.index}
    return best


def expected(m, idx):
    idx = pd.DatetimeIndex(idx)
    if m["kind"] == "ml":
        return pd.Series(m["mdl"].predict(_ml_features(idx)), index=idx).clip(lower=0)
    return pd.Series(_X(idx, m["th"], m["tc"]) @ m["b"], index=idx).clip(lower=0)


def accuracy(y, m):
    yb = y.reindex(m["idx"])
    e = yb - expected(m, yb.index)
    cv = float(np.sqrt((e ** 2).sum() / (len(e) - m["k"])) / yb.mean() * 100)
    nmbe = float(e.sum() / (len(e) - m["k"]) / yb.mean() * 100)
    rho = float(e.autocorr(1))
    return cv, nmbe, rho


def describe_model(p, m):
    res, unit = METER_INFO[p][1], RESOURCE[METER_INFO[p][1]]["unit"]
    b = list(m["b"]); base, offd = b[0], b[1]; i = 2
    s = [f"On a normal weekday in mild weather it uses about {base:,.0f} {unit}/day; on weekends and holidays "
         f"{abs(offd):,.0f} {'less' if offd < 0 else 'more'}."]
    if m["th"] is not None:
        s.append(f"For every 1 °C below {m['th']} °C it uses {b[i]:,.0f} {unit} more (heating)."); i += 1
    if m["tc"] is not None:
        s.append(f"For every 1 °C above {m['tc']} °C it uses {b[i]:,.0f} {unit} more (cooling).")
    return " ".join(s)


# ---- Which model predicts better? Test both on months they never saw (leave-one-month-out) ----
# For every meter, each baseline month is hidden in turn and predicted by (a) the change-point model and
# (b) gradient boosting (LightGBM, library defaults, same inputs + season). The model with the smaller
# typical miss on hidden months is used for predictions (alerts, verification). The change-point model is
# always kept too, because its slopes EXPLAIN the building (used for the accuracy grade and recommendations).
BASE_MONTHS = list(pd.period_range(BASE_START, BASE_END, freq="M"))
MODEL, CP_MODEL, MODEL_KIND, EXPECTED, OOS, MISS, ACC, MODEL_CHECK = {}, {}, {}, {}, {}, {}, [], []
for p in DAILY:
    yb = DAILY[p].loc[BASE_START:BASE_END].where(TEMP.notna())
    oos = {"cp": [], "ml": []}; miss = {"cp": {}, "ml": {}}
    for mo in BASE_MONTHS:
        a, b = mo.start_time, mo.end_time.normalize()
        hold = yb.loc[a:b].dropna()
        train = yb.copy(); train.loc[a:b] = np.nan
        days = DAYS[(DAYS >= a) & (DAYS <= b) & TEMP.notna().values]
        for kind in ("cp", "ml"):
            m_ = fit_model(train, kind)
            oos[kind].append(expected(m_, days))
            if len(hold) >= COMPLETE * len(pd.date_range(a, b)):
                e = expected(m_, hold.index).sum()
                miss[kind][a] = float((e - hold.sum()) / e * 100)
    rms = {k: float(np.sqrt(np.mean(np.square(list(miss[k].values()))))) for k in miss}
    MODEL_KIND[p] = "ml" if rms["ml"] < rms["cp"] else "cp"
    OOS[p] = pd.concat(oos[MODEL_KIND[p]])
    MISS[p] = miss[MODEL_KIND[p]]
    CP_MODEL[p] = fit_model(yb, "cp")
    MODEL[p] = CP_MODEL[p] if MODEL_KIND[p] == "cp" else fit_model(yb, "ml")
    EXPECTED[p] = expected(MODEL[p], DAYS[TEMP.notna().values])
    MODEL_CHECK.append({"point": p, "Meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}",
                        "Hidden months": len(miss["cp"]), "Change-point typical miss %": round(rms["cp"], 1),
                        "Gradient boosting typical miss %": round(rms["ml"], 1),
                        "Used for predictions": "Gradient boosting (ML)" if MODEL_KIND[p] == "ml" else "Change-point"})
    m = CP_MODEL[p]
    cv, nmbe, rho = accuracy(DAILY[p], m)
    grade = "Good" if cv <= 15 and abs(nmbe) <= 5 else ("OK" if cv <= 30 and abs(nmbe) <= 10 else "Not reliable")
    ACC.append({"point": p, "Meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}",
                "Baseline days": m["n"], "Typical daily error %": round(cv, 1), "Bias %": round(nmbe, 2),
                "Accuracy": grade, "Predictions by": MODEL_CHECK[-1]["Used for predictions"],
                "What the model learned": describe_model(p, m)})
ACC = pd.DataFrame(ACC)
MODEL_CHECK = pd.DataFrame(MODEL_CHECK)
RELIABLE = set(ACC[ACC.Accuracy != "Not reliable"].point)
print("Accuracy grade uses ASHRAE Guideline 14 limits on the change-point fit: Good = CV ≤ 15% & bias ≤ ±5% "
      "(monthly-data limit); OK = CV ≤ 30% & bias ≤ ±10% (hourly-data limit); otherwise Not reliable (no savings claimed).\n")
for _, r in ACC.iterrows():
    print(f"{r.Meter}\n   accuracy: {r.Accuracy} (typical daily error {r['Typical daily error %']}%) · predictions by: "
          f"{r['Predictions by']}\n   {r['What the model learned']}")
print("\nWhich model predicts hidden months better (typical monthly miss, lower is better):")
print(MODEL_CHECK.drop(columns="point").to_string(index=False))


## Cell 6 · Alerts and discoveries
After the baseline year, a meter is flagged when it uses **more** than its ghost twin by more than it ever did in the whole baseline year (each meter sets its own bar). Sustained **drops** are *discoveries* — good news, not counted until someone explains them.

In [ ]:
# ---- Unusual-change alerts ----
# Each meter sets its own bar from last year: we re-predict every baseline month with a model that did NOT
# see that month (leave-one-month-out), so the baseline errors are honest "surprises".
# ALERT     = after the baseline, a day (or 7-day stretch) used MORE than expected by more than the
#             biggest surprise of the whole baseline year.
# DISCOVERY = a 7-day stretch used LESS than expected by more than any 7 days of the baseline year
#             (good news nobody has explained yet — not counted as a saving until an action is linked).
# OOS (each baseline month predicted by a model that never saw it) comes from cell 5.
SCALES = {"1 day": 1, "7 days": 7}
RECORD, ALERTS, DISCOVERIES = {}, [], []
_expected_false = 0.0
for p in DAILY:
    y = DAILY[p]
    res_label, unit = RESOURCE[METER_INFO[p][1]]["label"], RESOURCE[METER_INFO[p][1]]["unit"]
    base_gap = (y - OOS[p]).loc[BASE_START:BASE_END]
    mon_gap = (y - EXPECTED[p]).loc[MON_START:]
    for name, w in SCALES.items():
        bg = base_gap.rolling(w, min_periods=w).sum().dropna()
        mg = mon_gap.rolling(w, min_periods=w).sum().dropna()
        hi, lo = float(bg.max()), float(bg.min())
        RECORD[(p, name)] = (hi, lo, len(bg))
        _expected_false += len(mg) / (len(bg) + 1)
        hits = mg[mg > hi]
        for d, v in hits.items():
            ALERTS.append({"point": p, "scale": name, "end": d, "start": d - pd.Timedelta(days=w - 1), "extra": v, "record": hi})
        if name == "7 days":
            for d, v in mg[mg < lo].items():
                DISCOVERIES.append({"point": p, "end": d, "start": d - pd.Timedelta(days=6), "less": -v, "record": -lo})


def _merge(df):
    """Join overlapping flagged windows into one event per meter."""
    out = []
    for p, g in df.sort_values("start").groupby("point", sort=False):
        cur = None
        for _, r in g.iterrows():
            if cur is not None and r.start <= cur["end"] + pd.Timedelta(days=1):
                cur["end"] = max(cur["end"], r.end); cur["rows"].append(r)
            else:
                if cur: out.append(cur)
                cur = {"point": p, "start": r.start, "end": r.end, "rows": [r]}
        if cur: out.append(cur)
    return out


def _qty(v, p):
    return f"{v:,.0f} {RESOURCE[METER_INFO[p][1]]['unit']}"


ALERT_TABLE = []
for ev in _merge(pd.DataFrame(ALERTS, columns=["point", "scale", "end", "start", "extra", "record"])):
    p = ev["point"]; worst = max(ev["rows"], key=lambda r: r.extra - r.record)
    y, e = DAILY[p].loc[ev["start"]:ev["end"]], EXPECTED[p].loc[ev["start"]:ev["end"]]
    extra = float((y - e).clip(lower=0).sum())
    ALERT_TABLE.append({
        "id": "", "meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}", "point": p,
        "from": ev["start"].date(), "to": ev["end"].date(), "extra use": _qty(extra, p),
        "extra %": round(extra / max(float(e.sum()), 1e-9) * 100, 1),
        "what happened": (f"{RESOURCE[METER_INFO[p][1]]['label']} at {METER_INFO[p][2]} used {_qty(worst.extra, p)} more than "
                          f"expected over {worst.scale} (ending {worst.end:%a %d %b}). The most it ever went over in the whole "
                          f"baseline year was {_qty(worst.record, p)}. This is new — worth a check."),
    })
ALERT_TABLE = pd.DataFrame(ALERT_TABLE, columns=["id", "meter", "point", "from", "to", "extra use", "extra %", "what happened"])
ALERT_TABLE["id"] = [f"AL-{i + 1:02d}" for i in range(len(ALERT_TABLE))]

DISCOVERY_TABLE = []
for ev in _merge(pd.DataFrame(DISCOVERIES, columns=["point", "end", "start", "less", "record"])):
    p = ev["point"]
    y, e = DAILY[p].loc[ev["start"]:ev["end"]], EXPECTED[p].loc[ev["start"]:ev["end"]]
    less = float((e - y).sum())
    DISCOVERY_TABLE.append({"meter": f"{RESOURCE[METER_INFO[p][1]]['icon']} {METER_INFO[p][2]}", "point": p,
                            "first seen": ev["rows"][0].end.date(), "from": ev["start"].date(), "to": ev["end"].date(),
                            "less than expected": _qty(less, p), "%": round(less / max(float(e.sum()), 1e-9) * 100, 1)})
DISCOVERY_TABLE = pd.DataFrame(DISCOVERY_TABLE, columns=["meter", "point", "first seen", "from", "to", "less than expected", "%"])
EXPECTED_FALSE_ALERTS = _expected_false

print(f"Alerts after the baseline: {len(ALERT_TABLE)}  (if nothing had changed, ≈{EXPECTED_FALSE_ALERTS:.1f} "
      f"flagged windows would be expected by chance)")
print(ALERT_TABLE[["id", "meter", "from", "to", "extra use", "extra %"]].to_string(index=False))
print(f"\nDiscoveries (sustained drops nobody has explained): {len(DISCOVERY_TABLE)}")
print(DISCOVERY_TABLE.drop(columns="point").to_string(index=False))


## Cell 7 · Waste
666 real hauling invoices — tonnes and cost per stream and building — and the **diversion rate** (compost + recycling ÷ total sorted). Bulky waste is shown but left out of the rate (Concordia: split unknown). Pickups with no weight are counted, not guessed.

In [ ]:
# ---- Waste: Concordia's real hauling invoices (Zero Waste program) ----
# Mass is in metric tonnes and Qty is a count of bins (Concordia API documentation, waste/invoices).
# Bulky waste is a mix of recycling and landfill in unknown proportions (Concordia's own definition),
# so it is reported separately and left out of the diversion rate. Pickups with no recorded weight are
# counted and shown — never filled in.
WASTE_TYPES = pd.read_csv(PATHS["WASTE_TYPE.csv"])
BUILDINGS = pd.read_csv(PATHS["BUILDING_LIST.csv"])
_w = pd.read_csv(PATHS["WASTE_INVOICES.csv"])
_w = _w.loc[:, [c for c in _w.columns if not c.startswith("Unnamed")]]
_w["Date"] = pd.to_datetime(_w["Date"], errors="coerce")
_w["tonnes"] = pd.to_numeric(_w["Mass"], errors="coerce")
_w["cost"] = pd.to_numeric(_w["Total_Cost"].astype(str).str.replace(r"[$,\s]", "", regex=True), errors="coerce")
WASTE = _w.dropna(subset=["Date"]).copy()
WASTE["month"] = WASTE["Date"].dt.to_period("M").dt.to_timestamp()
DIVERTED = ["Organics", "Paper", "Mixed_recycling"]                # sent to composting / recycling
STREAM_COLOR = {"Landfill": "#94A3B8", "Organics": "#22C55E", "Paper": "#38BDF8", "Mixed_recycling": "#A78BFA",
                "Bulky_waste": "#F59E0B"}
STREAM_LABEL = {"Landfill": "Landfill", "Organics": "Compost", "Paper": "Paper & cardboard",
                "Mixed_recycling": "Mixed recycling", "Bulky_waste": "Bulky (mixed, unknown split)"}

WASTE_MONTHLY = WASTE.pivot_table(index="month", columns="Waste_Type", values="tonnes", aggfunc="sum")
_weighed = WASTE[WASTE.tonnes.notna()].pivot_table(index="month", columns="Waste_Type", values="tonnes", aggfunc="count")
_full = _weighed.reindex(columns=["Landfill"] + DIVERTED).notna().all(axis=1)     # month has every stream weighed
_div = WASTE_MONTHLY.reindex(columns=DIVERTED).sum(axis=1)
_land = WASTE_MONTHLY.reindex(columns=["Landfill"]).sum(axis=1)
DIVERSION = (_div / (_div + _land) * 100).where(_full)
WASTE_BY_BUILDING = (WASTE.pivot_table(index="Building", columns="Waste_Type", values="tonnes", aggfunc="sum")
                     .fillna(0).round(1))
WASTE_BY_BUILDING["Total t"] = WASTE_BY_BUILDING.sum(axis=1)
WASTE_BY_BUILDING = WASTE_BY_BUILDING.sort_values("Total t", ascending=False)
WASTE_COST_PER_T = (WASTE[WASTE.tonnes > 0].groupby("Waste_Type").apply(lambda g: g.cost.sum() / g.tonnes.sum())).round(0)
NO_WEIGHT = int(WASTE.tonnes.isna().sum())

# before vs during the COVID closure, same 12-month baseline year as the meters
_b = WASTE[(WASTE.Date >= BASE_START) & (WASTE.Date <= BASE_END)]
WASTE_BASE = _b.groupby("Waste_Type").tonnes.sum().round(1)
_d = DIVERSION.loc[BASE_START:BASE_END].dropna()

print(f"Invoices: {len(WASTE)} ({WASTE.Date.min():%b %Y} → {WASTE.Date.max():%b %Y}), "
      f"{WASTE.Building.nunique()} buildings, {NO_WEIGHT} pickups with no recorded weight (left out, not guessed)")
print("\nTonnes in the baseline year:", WASTE_BASE.to_dict())
print(f"Diversion rate (compost + recycling ÷ those + landfill), months with every stream weighed: "
      f"baseline-year median {_d.median():.0f}%  ({len(_d)} months)")
print("\nCost per tonne ($ as invoiced):", WASTE_COST_PER_T.to_dict())
print("\nTonnes by building (all years):")
print(WASTE_BY_BUILDING.to_string())

# 12-month periods aligned with the meter baseline year (Mar → Feb)
WASTE["period"] = WASTE.Date.apply(lambda d: f"{d.year if d.month >= BASE_START.month else d.year - 1}–"
                                             f"{(d.year if d.month >= BASE_START.month else d.year - 1) + 1}")
_y = WASTE.pivot_table(index="period", columns="Waste_Type", values="tonnes", aggfunc="sum").fillna(0)
_y = _y[[c for c in ["Landfill"] + DIVERTED + ["Bulky_waste"] if c in _y]]
_cnt = WASTE.groupby("period").Date.agg(["min", "max"])
WASTE_YEARLY = _y.round(1)
_wc = WASTE[WASTE.tonnes > 0].pivot_table(index="period", columns="Waste_Type", values="tonnes", aggfunc="count")
_allw = _wc.reindex(index=_y.index, columns=["Landfill"] + DIVERTED).notna().all(axis=1)
for c in _y.columns:                                  # a stream never weighed in a period shows "—", not 0
    WASTE_YEARLY[c] = WASTE_YEARLY[c].where(_wc.reindex(index=_y.index).get(c, pd.Series(index=_y.index)).notna())
WASTE_YEARLY["Diversion %"] = (_y[DIVERTED].sum(axis=1) / (_y[DIVERTED].sum(axis=1) + _y["Landfill"]) * 100).round(0).where(_allw)
WASTE_YEARLY["Invoices from"] = _cnt["min"].dt.strftime("%b %Y")
WASTE_YEARLY["to"] = _cnt["max"].dt.strftime("%b %Y")
print("\nTonnes per 12-month period (Mar→Feb; partial periods marked by their invoice dates):")
print(WASTE_YEARLY.to_string())
print("Diversion % is shown only for periods where landfill, compost, paper and mixed recycling were all weighed.")
print("Note: 'Building' is where the bin or compactor sits — one compactor can serve several buildings.")


## Cell 7e · E-waste
Reads **your real asset register** (template provided) and routes every device the way India's **E-Waste (Management) Rules, 2022** require: keep in use, a *registered* refurbisher, or a *registered* recycler. Expected life comes from the CPCB EPR portal for each EEE code. Concordia's open data has no e-waste records, so nothing is shown until you upload a register — nothing is invented.

In [ ]:
# ---- E-waste: your real asset register → what to do with each device ----
# Nothing is invented. You upload a CSV of real devices (template written below). Rules follow India's
# E-Waste (Management) Rules, 2022: a bulk consumer (a university) may hand e-waste ONLY to a registered
# producer, refurbisher or recycler, and must keep records. "Expected life" is the average life the
# CPCB EPR portal lists for the device's EEE code (https://eprewastecpcb.in) — copy it into the CSV.
EWASTE_COLS = ["device_id", "device_type", "eee_code", "department", "location", "purchase_date",
               "expected_life_years", "condition", "weight_kg", "embodied_kgco2e"]
EWASTE_TEMPLATE = os.path.join(OUT_DIR, "ewaste_register_TEMPLATE.csv")
pd.DataFrame(columns=EWASTE_COLS).to_csv(EWASTE_TEMPLATE, index=False)
EWASTE_HELP = {
    "condition": "working / faulty / non-functional",
    "expected_life_years": "average life for the EEE code, from the CPCB EPR portal",
    "weight_kg": "optional — from the spec sheet or a scale",
    "embodied_kgco2e": "optional — manufacturer's product carbon footprint (PCF) sheet",
}
EWASTE_EOL_T_CO2E_PER_T = 0.02 / 0.907185    # EPA GHG Emission Factors Hub (2024), mixed electronics, recycled: 0.02 t CO2e / short ton
PATHS["ewaste_register.csv"] = find("ewaste_register.csv")


def ewaste_plan(df, today=None):
    """Flag devices and pick a route that is legal under the E-Waste (Management) Rules, 2022."""
    today = pd.Timestamp(today or datetime.date.today())
    d = df.copy()
    d["purchase_date"] = pd.to_datetime(d["purchase_date"], errors="coerce")
    for c in ["expected_life_years", "weight_kg", "embodied_kgco2e"]:
        d[c] = pd.to_numeric(d.get(c), errors="coerce")
    d["condition"] = d["condition"].astype(str).str.strip().str.lower()
    d["age_years"] = ((today - d["purchase_date"]).dt.days / 365.25).round(1)
    d["life_used_%"] = (d["age_years"] / d["expected_life_years"] * 100).round(0)

    def route(r):
        past = r["age_years"] >= r["expected_life_years"] if r["expected_life_years"] == r["expected_life_years"] else None
        if r["condition"] == "working" and past is False:
            return "Keep in use / redeploy", "working and within its expected life"
        if r["condition"] == "working":
            return "Registered refurbisher", "working but past its expected life — reuse before recycling"
        if r["condition"] == "faulty":
            return "Registered refurbisher (repair check)", "faulty — repair is assessed before recycling"
        if r["condition"] == "non-functional":
            return "Registered recycler", "non-functional — recover materials through an authorised recycler"
        return "Check record", "condition not recorded"
    rt = d.apply(route, axis=1, result_type="expand")
    d["route"], d["why"] = rt[0], rt[1]
    d["leaving campus"] = ~d["route"].isin(["Keep in use / redeploy", "Check record"])
    d["eol_kgco2e"] = (d["weight_kg"] / 1000 * EWASTE_EOL_T_CO2E_PER_T * 1000).where(d["leaving campus"])
    return d


EWASTE = None
if PATHS["ewaste_register.csv"]:
    EWASTE = ewaste_plan(pd.read_csv(PATHS["ewaste_register.csv"]))
    print(f"E-waste register: {len(EWASTE)} devices · {int(EWASTE['leaving campus'].sum())} to hand over to registered "
          f"refurbishers/recyclers · {EWASTE.loc[EWASTE['leaving campus'], 'weight_kg'].sum():,.0f} kg recorded")
    print(EWASTE.groupby("route").size().to_string())
else:
    print("No e-waste register uploaded yet. Concordia's open data has no e-waste records "
          f"(its invoices contain {int((WASTE.Waste_Type.str.lower() == 'e-waste').sum())} e-waste pickups).")
    print(f"→ Fill in the template with your department's real devices and upload it as ewaste_register.csv:\n   {EWASTE_TEMPLATE}")
    print("   Columns:", ", ".join(EWASTE_COLS))


## Cell 8 · Action register + "Did it work?" — contextual ghost twin + placebo-in-time
* **Ghost twin:** the building *without* the action, trained on the 12 months before and fed the **real** weather and calendar of every later day.
* **Placebo-in-time:** we pretend the action happened on every week of the year before (when nothing was done). Those fake "savings" show what noise looks like.
* **VERIFIED** only if the real change is outside the noise: 95% range excludes zero **and** p ≤ 5%. We count only the **low end**.

The register starts with one **ground-truth test** — the documented COVID-19 closure (13 March 2020) — to prove the engine on a real event. Approved recommendations and your own actions are added in the app.

🎤 *"A before/after dashboard says gas fell 40%. The ghost twin, fed the real spring weather, says that is normal — not proven, nothing counted."*

In [ ]:
# ---- Action register + "Did it work?" — contextual GHOST TWIN + PLACEBO-IN-TIME test ----
#   1. GHOST TWIN: a copy of the meter that never got the action. It is trained on the 12 months BEFORE the
#      action and then fed the REAL context of every later day (actual weather, weekday/holiday, season), so it
#      shows what the building would have used without the action. (IPMVP Option C / ASHRAE Guideline 14.)
#   2. saving = ghost twin − actual
#   3. PLACEBO-IN-TIME: we pretend the action happened on every week of the year before (when nothing was
#      done), hide that stretch, rebuild the ghost twin without it, and record the fake "saving". Those fakes
#      show what pure noise looks like for this meter.
#        • 95% range = ± t × (typical fake saving) × √(1 + 1/k), k = independent stretches in a year
#        • p-value  = share of fake savings at least as large as the real one (permutation test)
#   4. VERIFIED only if the whole 95% range excludes zero AND p ≤ 5%. We count the LOW end of the range.
ACTIONS_CSV = os.path.join(OUT_DIR, "actions.csv")
ACTION_COLS = ["action_id", "date", "point", "title", "type", "claimed_%", "source"]
if os.path.exists(ACTIONS_CSV):
    ACTIONS = pd.read_csv(ACTIONS_CSV, parse_dates=["date"])
else:
    # one real, documented event to start with (not a sustainability action — clearly labelled)
    ACTIONS = pd.DataFrame([{
        "action_id": "EV-01", "date": pd.Timestamp("2020-03-14"), "point": p,
        "title": "Ground-truth test — COVID-19: both campuses closed from midnight 13 March 2020",
        "type": "Ground-truth test (real, documented event) — proves the engine; not a sustainability saving",
        "claimed_%": np.nan,
        "source": "The Link, 13 Mar 2020: thelinknewspaper.ca/article/concordia-closed-until-march-29-due-to-covid-19-prevention"}
        for p in DAILY], columns=ACTION_COLS)
    ACTIONS["action_id"] = [f"EV-01{chr(97 + i)}" for i in range(len(ACTIONS))]
    ACTIONS.to_csv(ACTIONS_CSV, index=False)


def add_action(date, point, title, claimed_pct=None, source="logged in GreenLedger"):
    global ACTIONS
    aid = f"A-{len(ACTIONS) + 1:03d}"
    ACTIONS = pd.concat([ACTIONS, pd.DataFrame([{"action_id": aid, "date": pd.Timestamp(date), "point": point, "title": title,
                                                  "type": "Sustainability action", "claimed_%": claimed_pct,
                                                  "source": source}])], ignore_index=True)
    ACTIONS.to_csv(ACTIONS_CSV, index=False)
    return aid


WINDOWS = [28, 42, 56]          # 4, 6 or 8 whole weeks after the action (same weekday mix every time)
ELEC_KG_CO2_PER_KWH = 0.0005    # Hydro-Québec, 0.5 kg CO2/MWh for 2019 and 2020 ("CO2 emissions and Hydro-Québec electricity 1990-2020")
GAS_KG_CO2_PER_M3 = 1.926       # Québec MELCC GHG quantification guide, Table 5 (natural gas, CO2)
GAS_M3_PER_METER_UNIT = None    # Concordia's API documents m³. Sanity check (cell 10): at 1 unit = 1 m³ the EV+GM+MB meter
                                # alone would emit ~57,000 t CO₂/yr — far above everything else measured — so the unit is
                                # treated as unconfirmed. Set to 1.0 once Facilities confirms m³ (the app has a switch).


def co2_kg(point, amount):
    res = METER_INFO[point][1]
    if res == "electricity":
        return amount * ELEC_KG_CO2_PER_KWH
    if res == "gas" and GAS_M3_PER_METER_UNIT:
        return amount * GAS_M3_PER_METER_UNIT * GAS_KG_CO2_PER_M3
    return None


ALPHA = 0.05                       # 95% confidence — statistical convention
PLACEBO_STEP = 7                   # fake action dates one week apart: every date has the same weekday mix


def placebo_in_time(y_base, window, kind="cp"):
    """Fake action on every week of the baseline year → the 'saving' the ghost twin finds when nothing was done."""
    start, end = y_base.index.min(), y_base.index.max()
    fakes, d = [], start
    while d + pd.Timedelta(days=window - 1) <= end:
        b = d + pd.Timedelta(days=window - 1)
        hold = y_base.loc[d:b].dropna()
        if len(hold) >= COMPLETE * window:
            train = y_base.copy(); train.loc[d:b] = np.nan
            e = expected(fit_model(train, kind), hold.index)
            fakes.append(float((e.sum() - hold.sum()) / e.sum() * 100))
        d += pd.Timedelta(days=PLACEBO_STEP)
    return np.array(fakes)


def range_from_misses(misses, k=None):
    """Prediction interval from real misses. k = number of INDEPENDENT stretches (overlapping fakes share data)."""
    k = k or len(misses)
    typical = float(np.sqrt(np.mean(np.square(misses))))       # typical miss (root-mean-square, includes any bias)
    return float(stats.t.ppf(1 - ALPHA / 2, k - 1) * typical * np.sqrt(1 + 1 / k)), typical


def verify(point, date, window=42, other_events=None):
    T0 = pd.Timestamp(date).normalize()
    res, unit = METER_INFO[point][1], RESOURCE[METER_INFO[point][1]]["unit"]
    y = DAILY[point].where(TEMP.notna())
    base = y.loc[T0 - pd.Timedelta(days=365): T0 - pd.Timedelta(days=1)]
    post = y.loc[T0: T0 + pd.Timedelta(days=window - 1)]
    out = {"point": point, "date": T0, "window": window, "unit": unit}
    if post.notna().sum() < COMPLETE * window:
        return {**out, "verdict": "WAITING FOR DATA",
                "why": f"Needs {window} days of good readings after the action; {int(post.notna().sum())} available."}
    if base.notna().sum() < COMPLETE * 365:
        return {**out, "verdict": "NOT ENOUGH HISTORY",
                "why": f"Needs 12 months of good readings (with weather) before the action; "
                       f"{int(base.notna().sum())} days available."}
    kind = MODEL_KIND[point]
    cp = fit_model(base, "cp")
    cv, nmbe, rho = accuracy(base, cp)                       # ASHRAE Guideline 14 grade on the change-point fit
    ghost = cp if kind == "cp" else fit_model(base, kind)   # the meter's better predictor (cell 5)
    fakes = placebo_in_time(base, window, kind)
    k_ind = 365 // window
    if len(fakes) < k_ind:
        return {**out, "verdict": "NOT ENOUGH HISTORY", "why": "Too few complete weeks in the baseline year for the placebo test."}
    half, typical = range_from_misses(fakes, k_ind)
    yp = post.dropna()
    exp_ = expected(ghost, yp.index)
    saving = float((exp_.sum() - yp.sum()) / exp_.sum() * 100)
    lo, hi = saving - half, saving + half
    p = float((1 + np.sum(np.abs(fakes) >= abs(saving))) / (len(fakes) + 1))
    pre = y.loc[T0 - pd.Timedelta(days=window): T0 - pd.Timedelta(days=1)].mean()
    naive = float((pre - yp.mean()) / pre * 100)
    reliable = cv <= 30 and abs(nmbe) <= 10
    if not reliable:
        verdict = "MODEL NOT ACCURATE ENOUGH"
    elif lo > 0 and p <= ALPHA:
        verdict = "VERIFIED SAVING"
    elif hi < 0 and p <= ALPHA:
        verdict = "VERIFIED INCREASE"
    else:
        verdict = "NOT PROVEN"
    counted = max(0.0, lo / 100 * float(exp_.sum())) if verdict == "VERIFIED SAVING" else 0.0
    warn = []
    if other_events is not None:
        clash = other_events[(other_events.point == point) & (other_events.date < T0) &
                             (other_events.date >= T0 - pd.Timedelta(days=365))]
        if len(clash):
            warn.append("Another logged event falls inside this action's 12-month baseline "
                        f"({', '.join(clash.title.str[:40])}). The baseline is not 'normal', so treat this result with care.")
    return {**out, "verdict": verdict, "saving_%": saving, "low_%": lo, "high_%": hi, "p": p, "naive_%": naive,
            "expected": float(exp_.sum()), "actual": float(yp.sum()), "counted": counted,
            "counted_co2_kg": co2_kg(point, counted) if counted else 0.0, "cv": cv, "rho": rho,
            "fakes": fakes, "typical_miss": typical, "kind": kind, "k_ind": k_ind,
            "days": len(yp), "model": ghost, "series_exp": exp_, "series_act": yp, "warnings": warn}


def explain(r):
    """One plain-English paragraph a judge can read."""
    name = f"{RESOURCE[METER_INFO[r['point']][1]]['label']} at {METER_INFO[r['point']][2]}"
    if "saving_%" not in r:
        return f"{r['verdict']}: {r['why']}"
    u = r["unit"]
    txt = (f"Over the {r['days']} days after {r['date']:%d %b %Y}, {name} used {r['actual']:,.0f} {u}. "
           f"Based on the 12 months before — adjusted for the actual weather and weekdays/holidays — it would have used "
           f"about {r['expected']:,.0f} {u} — that is its ghost twin ({'gradient-boosting' if r['kind'] == 'ml' else 'change-point'} model, "
           f"fed the real weather and calendar). That is a {abs(r['saving_%']):.1f}% {'drop' if r['saving_%'] > 0 else 'rise'}. "
           f"Placebo test: we pretended the action happened on {len(r['fakes'])} other dates last year; those fake "
           f"'savings' were typically {r['typical_miss']:.1f}%, so the 95% range is {r['low_%']:.1f}% to {r['high_%']:.1f}% "
           f"and only {r['p'] * 100:.0f}% of fakes were this large (p = {r['p']:.3f}). "
           f"A simple before/after comparison would have said {r['naive_%']:.1f}%. ")
    if r["verdict"] == "VERIFIED SAVING":
        txt += f"Both tests pass, so we count the LOW end: {r['counted']:,.0f} {u}."
        if r["counted_co2_kg"]:
            txt += f" That avoids about {r['counted_co2_kg']:,.0f} kg CO₂ on Québec's hydro-powered grid."
    elif r["verdict"] == "VERIFIED INCREASE":
        txt += "Use went UP beyond normal variation."
    elif r["verdict"] == "MODEL NOT ACCURATE ENOUGH":
        txt += (f"This meter's model error ({r['cv']:.0f}%) is above the ASHRAE Guideline 14 limit, so no saving is "
                f"claimed — only the raw numbers are shown.")
    else:
        txt += "At least one test fails, so the change could be normal variation. Nothing is counted."
    for w in r["warnings"]:
        txt += " ⚠ " + w
    return txt


VERIFIED = {}
for _, a in ACTIONS.iterrows():
    VERIFIED[a.action_id] = verify(a.point, a.date, 42, ACTIONS[ACTIONS.action_id != a.action_id])
    print(f"[{a.action_id}] {RESOURCE[METER_INFO[a.point][1]]['icon']} {METER_INFO[a.point][2]}: {VERIFIED[a.action_id]['verdict']}")
    print("    " + explain(VERIFIED[a.action_id]))


## Cell 9 · Can you trust it? (the fair test)
Normal months where we **know nothing was done**, each hidden from the model: how often does each method report a "saving"?

🎤 *"On months where nothing happened, a simple dashboard reported savings almost half the time. GreenLedger almost never does."*

In [ ]:
# ---- "Can you trust it?" — a fair test where we KNOW nothing was done ----
# For each of the 12 normal months in the baseline year, we pretend an action started on the 1st,
# hide that month from the model, and ask both methods whether it "saved" anything.
# A simple before/after dashboard (this month vs the 30 days before) vs. GreenLedger (cell 8 method).
TRUST = []
for p in RELIABLE:
    y = DAILY[p].where(TEMP.notna())
    miss = MISS[p]                                           # each month predicted by a model that never saw it (cell 5)
    for a, gl in miss.items():
        others = np.array([v for k, v in miss.items() if k != a])       # the range never uses the month being tested
        half, _ = range_from_misses(others)              # monthly stretches don't overlap → k = number of months
        post = y.loc[a:a + pd.offsets.MonthEnd(0)].dropna()
        prev = y.loc[a - pd.Timedelta(days=30): a - pd.Timedelta(days=1)].mean()
        TRUST.append({"point": p, "month": a, "naive_%": float((prev - post.mean()) / prev * 100),
                      "gl_%": gl, "gl_low": gl - half, "gl_high": gl + half})
TRUST = pd.DataFrame(TRUST)
TT = TRUST.dropna(subset=["gl_%", "naive_%"])
TRUST_SUMMARY = {
    "tests": len(TT),
    "naive_saving": int((TT["naive_%"] > 0).sum()),
    "naive_big": int((TT["naive_%"] >= 5).sum()),
    "gl_saving": int((TT.gl_low > 0).sum()),
    "gl_increase": int((TT.gl_high < 0).sum()),
}
TRUST_SUMMARY["naive_avg_abs"] = round(float(TT["naive_%"].abs().mean()), 1)
TRUST_SUMMARY["gl_avg_abs"] = round(float(TT["gl_%"].abs().mean()), 1)
print(f"{TRUST_SUMMARY['tests']} normal months on meters with an accurate model — nothing was done in any of them.")
print(f"  A simple before/after dashboard reported a 'saving' in {TRUST_SUMMARY['naive_saving']} of them "
      f"({TRUST_SUMMARY['naive_big']} of 5% or more); average apparent change {TRUST_SUMMARY['naive_avg_abs']}%.")
print(f"  GreenLedger verified a saving in {TRUST_SUMMARY['gl_saving']} and an increase in {TRUST_SUMMARY['gl_increase']}; "
      f"average apparent change {TRUST_SUMMARY['gl_avg_abs']}%.")


## Cell 9r · 💡 Recommendations — the optimisation layer
Turns each building's own data into actions, plus a **what-if simulator** (you pick: °C of setpoint change, % less on weekends, % less always-on load → kWh, m³, CO₂ from the building's own model and last year's real weather): weekend/holiday schedules (benchmarked against another Concordia meter that already does better), heating/cooling sensitivity per 1 °C (with last year's real weather), always-on base load, gas used regardless of weather, missing recycling/compost bins, cheaper waste streams. Every size is a what-if from real numbers — **potential, not verified** — and a person approves or rejects each one. Approved actions go into the register, so the ghost twin verifies them later.

🎤 *"Where's the optimisation? Here: recommendations sized from real data → a person approves → the ghost twin proves it. Recommend, act, verify."*

In [ ]:
# ---- 💡 Recommendations: what to do next — read straight from each building's own data ----
# No invented targets. Every size below is either
#   • an internal benchmark — "if this meter did what another Concordia meter ALREADY does", or
#   • a per-unit sensitivity — "each 1 °C / each 1 % is worth …", computed with last year's real weather.
# Each recommendation is marked "potential — not verified": once it is done, log it in the action register
# and GreenLedger verifies it after 4–8 weeks (cell 8). That closes the loop: recommend → act → verify.
LANDFILL_T_CO2E_PER_T = 0.58 / 0.907185   # EPA GHG Emission Factors Hub (2024), mixed MSW landfilled: 0.58 t CO2e per short ton
RECS = []


def _rec(group, area, where, finding, size, size_value, unit, action, point=None):
    RECS.append({"priority": group, "area": area, "where": where, "finding": finding, "potential (not verified)": size,
                 "_value": size_value, "unit": unit, "next step": action, "_point": point})


# 1) open alerts → fix now
for _, a in ALERT_TABLE.iterrows():
    _rec("1 · Fix now", RESOURCE[METER_INFO[a.point][1]]["label"], METER_INFO[a.point][2],
         a["what happened"], f"{a['extra use']} extra already used", np.nan, "", "Check what was running; log the fix as an action.", a.point)

BY_RES = {}
for p in RELIABLE:
    BY_RES.setdefault(METER_INFO[p][1], []).append(p)
BASE_DAYS = pd.date_range(BASE_START, BASE_END, freq="D")
N_OFF = int(OFF_DAY.loc[BASE_START:BASE_END].sum())

# 2) weekend / holiday schedules — internal benchmark
OFFDAY = {}
for p in RELIABLE:
    y = DAILY[p].loc[BASE_START:BASE_END]
    off = OFF_DAY.loc[BASE_START:BASE_END]
    W, O = float(y[~off].mean()), float(y[off].mean())
    OFFDAY[p] = {"weekday": W, "offday": O, "cut": (W - O) / W * 100}
for res, pts in BY_RES.items():
    best = max(pts, key=lambda q: OFFDAY[q]["cut"])
    for p in pts:
        d, bd = OFFDAY[p], OFFDAY[best]
        unit = RESOURCE[res]["unit"]
        if p == best:
            continue
        gap = (bd["cut"] - d["cut"]) / 100 * d["weekday"] * N_OFF
        _rec("2 · Quick win", RESOURCE[res]["label"], METER_INFO[p][2],
             f"On weekends and holidays it uses {abs(d['cut']):.0f}% {'less' if d['cut'] >= 0 else 'MORE'} than on weekdays. "
             f"{METER_INFO[best][2]} already cuts {bd['cut']:.0f}%.",
             f"≈ {gap:,.0f} {unit}/year if it matched {METER_INFO[best][2]}", gap, unit,
             "Review HVAC and lighting schedules for weekends and holidays.", p)

# 3) temperature sensitivity — each 1 °C of setpoint / balance temperature, with last year's real weather
for p in RELIABLE:
    m, res = CP_MODEL[p], METER_INFO[p][1]
    unit = RESOURCE[res]["unit"]
    t = TEMP.loc[BASE_START:BASE_END].dropna()
    exp_year = float(expected(m, t.index).sum())
    i = 2
    if m["th"] is not None:
        sh = m["b"][i]; i += 1
        per_deg = float(sh * np.clip(m["th"] - t, 0, 1).sum())
        share = float(sh * np.maximum(0, m["th"] - t).sum()) / exp_year * 100
        _rec("3 · Plan", RESOURCE[res]["label"], METER_INFO[p][2],
             f"{share:.0f}% of its use is driven by cold weather (starts below {m['th']} °C; +{sh:,.0f} {unit} per °C colder).",
             f"≈ {per_deg:,.0f} {unit}/year for each 1 °C lower heating balance point", per_deg, unit,
             "Lower heating setpoints / add night & weekend setbacks; then verify.", p)
    if m["tc"] is not None:
        sc = m["b"][i]
        per_deg = float(sc * np.clip(t - m["tc"], 0, 1).sum())
        share = float(sc * np.maximum(0, t - m["tc"]).sum()) / exp_year * 100
        _rec("3 · Plan", RESOURCE[res]["label"], METER_INFO[p][2],
             f"{share:.0f}% of its use is driven by warm weather (starts above {m['tc']} °C; +{sc:,.0f} {unit} per °C warmer).",
             f"≈ {per_deg:,.0f} {unit}/year for each 1 °C higher cooling balance point", per_deg, unit,
             "Raise cooling setpoints / widen the dead-band; then verify.", p)

# 4) base load (electricity, hourly data) — what runs 24/7
BASELOAD = {}
for p in [q for q in RELIABLE if METER_INFO[q][1] == "electricity"]:
    h = HOURLY[p].loc[BASE_START:BASE_END + pd.Timedelta(hours=23)]
    g = h.groupby(h.index.normalize())
    ok = g.count() >= COMPLETE * 24
    mn, mean = g.min()[ok], g.mean()[ok]
    kw = float(mn.median())
    BASELOAD[p] = {"kW": kw, "share": float((mn / mean).median() * 100), "kWh_year": kw * 8760}
    _rec("3 · Plan", "Electricity", METER_INFO[p][2],
         f"Never drops below about {kw:,.0f} kW, even at night — {BASELOAD[p]['share']:.0f}% of its average demand runs around the clock.",
         f"≈ {kw * 8760 / 100:,.0f} kWh/year for each 1% cut in this always-on load", kw * 8760 / 100, "kWh",
         "Night walk-through: ventilation, servers, lab equipment left on.", p)

# 5) gas used regardless of weather
for p in [q for q in RELIABLE if METER_INFO[q][1] == "gas"]:
    m = CP_MODEL[p]
    t = TEMP.loc[BASE_START:BASE_END].dropna()
    base_part = float((m["b"][0] + m["b"][1] * OFF_DAY.reindex(t.index).astype(float)).sum())
    share = base_part / float(expected(m, t.index).sum()) * 100
    _rec("3 · Plan", "Natural gas", METER_INFO[p][2],
         f"{share:.0f}% of its gas is used whatever the weather (≈ {m['b'][0]:,.0f} m³ on a mild weekday).",
         f"≈ {base_part / 100:,.0f} m³/year for each 1% cut in weather-independent gas", base_part / 100, "m³",
         "Check boiler lock-out in warm weather and hot-water schedules.", p)

# 6) waste — streams missing where landfill is invoiced, cost and CO2e per tonne moved
_wb = WASTE[WASTE.tonnes > 0].pivot_table(index="Building", columns="Waste_Type", values="tonnes", aggfunc="sum").fillna(0)
_wbase = (WASTE[(WASTE.Date >= BASE_START) & (WASTE.Date <= BASE_END) & (WASTE.Waste_Type == "Landfill")]
          .groupby("Building").tonnes.sum())
for bld, r in _wb.iterrows():
    if r.get("Landfill", 0) > 0:
        missing = [STREAM_LABEL[s_].lower() for s_ in DIVERTED if r.get(s_, 0) == 0]
        if missing:
            lt = float(_wbase.get(bld, 0))
            _rec("2 · Quick win", "Waste", f"{bld} bins",
                 f"{r['Landfill']:,.0f} t to landfill invoiced here (all years), but no {' / '.join(missing)} stream at this location.",
                 f"{lt:,.0f} t landfilled here in the baseline year ≈ {lt * LANDFILL_T_CO2E_PER_T:,.0f} t CO₂e; "
                 f"each tonne diverted avoids ≈ {LANDFILL_T_CO2E_PER_T:.2f} t CO₂e", lt, "t",
                 "Add the missing bins next to the landfill bins; track the diversion rate.")
_cheap = WASTE_COST_PER_T.drop(labels=["Landfill", "Bulky_waste"], errors="ignore")
_cheap = _cheap[_cheap < WASTE_COST_PER_T.get("Landfill", np.inf)]
if len(_cheap):
    s = _cheap.idxmin()
    diff = WASTE_COST_PER_T["Landfill"] - _cheap[s]
    _rec("2 · Quick win", "Waste", "campus", f"Landfill costs ${WASTE_COST_PER_T['Landfill']:,.0f}/t; {STREAM_LABEL[s].lower()} costs "
         f"${_cheap[s]:,.0f}/t (from the invoices).", f"${diff:,.0f} saved per tonne moved from landfill to {STREAM_LABEL[s].lower()}",
         diff, "$/t", "Better sorting signs / bin placement for recyclables.")
LANDFILL_BASE_T = float(WASTE_BASE.get("Landfill", 0))
LANDFILL_BASE_CO2E = LANDFILL_BASE_T * LANDFILL_T_CO2E_PER_T

RECS = pd.DataFrame(RECS).sort_values(["priority", "area", "_value"], ascending=[True, True, False]).reset_index(drop=True)
_wb_drop = RECS[(RECS.area == "Waste") & (RECS._value < 1)].index      # locations with no landfill in the baseline year
RECS = RECS.drop(_wb_drop).reset_index(drop=True)
RECS.insert(0, "id", [f"R-{i + 1:02d}" for i in range(len(RECS))])

# ---- human in the loop: every recommendation waits for a person to approve or reject it (with a reason) ----
DECISIONS_CSV = os.path.join(OUT_DIR, "recommendation_decisions.csv")
DECISIONS = pd.read_csv(DECISIONS_CSV) if os.path.exists(DECISIONS_CSV) else pd.DataFrame(
    columns=["time", "rec_id", "decision", "by", "reason", "action_id"])


def decide_rec(rec_id, decision, by="", reason="", action_date=None):
    """Approve → the action goes into the register (dated) so the ghost twin can verify it later. Reject → reason kept."""
    global DECISIONS
    r = RECS.set_index("id").loc[rec_id]
    aid = ""
    if decision == "approved" and isinstance(r["_point"], str) and action_date:
        aid = add_action(action_date, r["_point"], f"{r['next step']} ({rec_id})", None, f"recommendation {rec_id}")
    DECISIONS = pd.concat([DECISIONS, pd.DataFrame([{"time": datetime.datetime.now().strftime("%Y-%m-%d %H:%M"),
                                                     "rec_id": rec_id, "decision": decision, "by": by, "reason": reason,
                                                     "action_id": aid}])], ignore_index=True)
    DECISIONS.to_csv(DECISIONS_CSV, index=False)
    return aid
TOP_RECS = pd.concat([RECS[RECS.priority.str.startswith("1")].head(1),
                      RECS[RECS.priority.str.startswith("2") & (RECS.area != "Waste")].head(1),
                      RECS[(RECS.priority.str.startswith("3")) & (RECS.area == "Electricity")].head(1)])
print(f"{len(RECS)} recommendations ({RECS.priority.value_counts().sort_index().to_dict()})")
print(f"Landfill in the baseline year: {LANDFILL_BASE_T:,.0f} t ≈ {LANDFILL_BASE_CO2E:,.0f} t CO₂e (EPA factor)\n")
for _, r in RECS.iterrows():
    print(f"{r.id} [{r.priority}] {r.area} · {r['where']}\n    {r.finding}\n    → {r['potential (not verified)']} · {r['next step']}")


# ---- 🎚️ What-if simulator: YOU choose the scenario; the numbers come from each building's own model ----
# Uses the change-point slopes and LAST YEAR'S REAL WEATHER, campus meters only (no double counting:
# EV sits inside EV+GM+MB+TD, GM inside EV+GM+MB). These are projections — not savings until verified.
_T_BASE = TEMP.loc[BASE_START:BASE_END].dropna()
_OFF_BASE = OFF_DAY.reindex(_T_BASE.index).astype(bool)


def _slopes(m):
    b = list(m["b"]); i = 2; sh = sc = 0.0
    if m["th"] is not None:
        sh = b[i]; i += 1
    if m["tc"] is not None:
        sc = b[i]
    return sh, sc


def what_if(heat_down=0.0, cool_up=0.0, offday_cut=0.0, base_cut=0.0):
    """heat_down: °C lower heating balance point · cool_up: °C higher cooling balance point ·
    offday_cut: % less use on weekends/holidays · base_cut: % less always-on electricity."""
    rows = []
    for res in ("electricity", "gas"):
        for p in CAMPUS_TOTAL[res]:
            if p not in RELIABLE:
                continue
            m = CP_MODEL[p]; sh, sc = _slopes(m); t = _T_BASE
            heat = sh * float((np.maximum(0, m["th"] - t) - np.maximum(0, m["th"] - heat_down - t)).sum()) if m["th"] is not None else 0.0
            cool = sc * float((np.maximum(0, t - m["tc"]) - np.maximum(0, t - m["tc"] - cool_up)).sum()) if m["tc"] is not None else 0.0
            offd = float(expected(m, t.index)[_OFF_BASE.values].sum()) * offday_cut / 100
            base = BASELOAD[p]["kWh_year"] * base_cut / 100 if p in BASELOAD else 0.0
            rows.append({"point": p, "meter": METER_INFO[p][2], "resource": res, "unit": RESOURCE[res]["unit"],
                         "heating": heat, "cooling": cool, "weekends": offd, "base load": base,
                         "total": heat + cool + offd + base})
    W = pd.DataFrame(rows)
    kwh = float(W[W.resource == "electricity"].total.sum()); m3 = float(W[W.resource == "gas"].total.sum())
    co2_e = kwh * ELEC_KG_CO2_PER_KWH / 1000
    co2_g = m3 * (GAS_M3_PER_METER_UNIT or 0) * GAS_KG_CO2_PER_M3 / 1000 if GAS_M3_PER_METER_UNIT else None
    return W, kwh, m3, co2_e, co2_g


_W, _k, _m, _ce, _cg = what_if(1, 1, 10, 1)
print(f"\nWhat-if example (1 °C lower heating point, 1 °C higher cooling point, 10% less on weekends, 1% less always-on load), "
      f"with last year's weather: ≈ {_k / 1000:,.0f} MWh electricity and ≈ {_m:,.0f} m³ (meter units) of gas per year "
      f"→ {_ce:,.1f} t CO₂ from electricity. Projection only — verify after doing it.")


## Cell 10 · 🌍 Carbon report
GHG Protocol scopes with sourced factors: Scope 2 electricity (Hydro-Québec), Scope 3 landfill and e-waste (US EPA Emission Factors Hub), Scope 1 gas withheld until the meter unit is confirmed. Data gaps are not filled.

🎤 *"In Québec, landfill waste out-emits all metered electricity — on India's grid the same electricity would be ~1,400× more CO₂. Where you are changes what to fix first."*

In [ ]:
# ---- 🌍 Carbon report (GHG Protocol scopes) — only what can be measured and sourced ----
# Scope 1  natural gas burned on campus  → needs the gas meter unit confirmed (cell 8); until then: not computed
# Scope 2  purchased electricity         → metered kWh × Hydro-Québec 0.5 kg CO2/MWh (2019, 2020)
# Scope 3  waste sent to landfill        → invoiced tonnes × EPA landfill factor (cell 9r)
#          e-waste handed over           → kg × EPA mixed-electronics factor (cell 7e), if a register is uploaded
# Days with no good reading are NOT filled in; each line shows how complete the data was.
def _period(d):
    y0 = d.year if d.month >= BASE_START.month else d.year - 1
    return f"{pd.Timestamp(y0, BASE_START.month, 1):%b %Y} → {pd.Timestamp(y0 + 1, BASE_START.month, 1) - pd.Timedelta(days=1):%b %Y}"


CARBON_ROWS = []
for p in CAMPUS_TOTAL["electricity"]:
    s = DAILY[p].dropna()
    for per, g in s.groupby(s.index.map(_period)):
        days_in = len(pd.date_range(*[pd.Timestamp(x) for x in (g.index.min(), g.index.max())]))
        CARBON_ROWS.append({"period": per, "scope": "Scope 2 · electricity", "source": METER_INFO[p][2],
                            "amount": float(g.sum()), "unit": "kWh", "t CO2e": float(g.sum()) * ELEC_KG_CO2_PER_KWH / 1000,
                            "data": f"{len(g)} good days"})
for p in CAMPUS_TOTAL["gas"]:
    s = DAILY[p].dropna()
    for per, g in s.groupby(s.index.map(_period)):
        co2 = co2_kg(p, float(g.sum()))
        CARBON_ROWS.append({"period": per, "scope": "Scope 1 · natural gas", "source": METER_INFO[p][2],
                            "amount": float(g.sum()), "unit": "meter units (documented m³)",
                            "t CO2e": co2 / 1000 if co2 is not None else np.nan,
                            "data": f"{len(g)} good days" + ("" if co2 is not None else " · unit not confirmed → CO₂ not computed")})
_lw = WASTE[(WASTE.Waste_Type == "Landfill") & WASTE.tonnes.notna()]
for per, g in _lw.groupby(_lw.Date.map(_period)):
    CARBON_ROWS.append({"period": per, "scope": "Scope 3 · waste to landfill", "source": "hauling invoices",
                        "amount": float(g.tonnes.sum()), "unit": "t", "t CO2e": float(g.tonnes.sum()) * LANDFILL_T_CO2E_PER_T,
                        "data": f"{len(g)} weighed pickups"})
if EWASTE is not None and EWASTE["leaving campus"].any():
    kg = float(EWASTE.loc[EWASTE["leaving campus"], "weight_kg"].sum())
    CARBON_ROWS.append({"period": "register", "scope": "Scope 3 · e-waste", "source": "asset register",
                        "amount": kg, "unit": "kg", "t CO2e": kg / 1000 * EWASTE_EOL_T_CO2E_PER_T, "data": "end-of-life only"})
CARBON = pd.DataFrame(CARBON_ROWS)
CARBON_BY_PERIOD = CARBON.pivot_table(index="period", columns="scope", values="t CO2e",
                                      aggfunc=lambda s: s.sum(min_count=1)).round(1)
INDIA_GRID_KG_PER_KWH = 0.710   # CEA CO2 Baseline Database v21.0 (India) — for comparison only

AVOIDED = pd.DataFrame([{"action": k, "meter": METER_INFO[v["point"]][2], "counted": v.get("counted", 0), "unit": v["unit"],
                         "kg CO2e avoided": v.get("counted_co2_kg") or 0.0, "verdict": v["verdict"]}
                        for k, v in VERIFIED.items()])
CARBON_FACTORS = pd.DataFrame([
    ("Electricity (Scope 2)", f"{ELEC_KG_CO2_PER_KWH * 1000:.1f} kg CO₂ / MWh", "Hydro-Québec — CO2 emissions and Hydro-Québec electricity 1990–2020"),
    ("India grid (comparison only)", f"{INDIA_GRID_KG_PER_KWH} kg CO₂ / kWh", "CEA CO2 Baseline Database v21.0"),
    ("Natural gas (Scope 1)", f"{GAS_KG_CO2_PER_M3} kg CO₂ / m³ (used once meter unit is confirmed)", "Québec MELCC GHG quantification guide, Table 5"),
    ("Landfilled mixed waste (Scope 3)", f"{LANDFILL_T_CO2E_PER_T:.3f} t CO₂e / t", "US EPA GHG Emission Factors Hub 2024 (0.58 t / short ton)"),
    ("E-waste, recycled (Scope 3)", f"{EWASTE_EOL_T_CO2E_PER_T:.3f} t CO₂e / t", "US EPA GHG Emission Factors Hub 2024 (0.02 t / short ton)"),
], columns=["What", "Factor", "Source"])
print("t CO₂e by 12-month period (blank = not measured / not computable):")
print(CARBON_BY_PERIOD.to_string())
_e = CARBON[(CARBON.scope.str.contains("electricity")) & (CARBON.period == _period(BASE_START))].amount.sum()
print(f"\nThe surprise for judges: Québec's grid is almost all hydro, so {_e / 1e6:,.1f} GWh of metered electricity in the "
      f"baseline year is only {_e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t CO₂ — landfill waste is the biggest measured source. "
      f"On India's grid (CEA, {INDIA_GRID_KG_PER_KWH} kg/kWh) the same electricity would be {_e * INDIA_GRID_KG_PER_KWH / 1000:,.0f} t CO₂.")

# ---- Gas sanity check: the documented unit vs everything else we measured ----
FT3_TO_M3 = 0.0283168                       # exact unit conversion (1 cubic foot in m³)
_gb = sum(float(DAILY[p].loc[BASE_START:BASE_END].sum()) for p in CAMPUS_TOTAL["gas"])
GAS_DOC_T = _gb * GAS_KG_CO2_PER_M3 / 1000             # if 1 meter unit = 1 m³ (as documented)
GAS_FT3_T = _gb * FT3_TO_M3 * GAS_KG_CO2_PER_M3 / 1000   # if the meter were actually in cubic feet
_land = float(_cb_land := CARBON[(CARBON.period == _period(BASE_START)) & CARBON.scope.str.contains("landfill")]["t CO2e"].sum())
print(f"\nGas, baseline year: {_gb:,.0f} meter units. As documented (m³) → {GAS_DOC_T:,.0f} t CO₂; "
      f"even if the unit were cubic feet → {GAS_FT3_T:,.0f} t CO₂. Either way gas is the biggest source "
      f"(landfill {_land:,.0f} t, electricity {_e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t) — so gas is the lever, and its unit "
      f"must be confirmed before a tonne figure is published.")


def carbon_by_period(include_gas_documented=False):
    C = CARBON.copy()
    if include_gas_documented and not GAS_M3_PER_METER_UNIT:
        g = C.scope.str.contains("gas")
        C.loc[g, "t CO2e"] = C.loc[g, "amount"] * GAS_KG_CO2_PER_M3 / 1000
    return C.pivot_table(index="period", columns="scope", values="t CO2e", aggfunc=lambda s: s.sum(min_count=1)).round(1)


## Cell 11 · The six sentences for your pitch
All computed above from real data. Read these out on stage.

In [ ]:
# ---- The numbers to say to the judges (all computed above from real data) ----
PARAMETERS = pd.DataFrame([
    ("Data", "Concordia University open data (Facilities), CC BY 4.0", "Real meters, real invoices — nothing simulated"),
    ("Day counts if", "≥ 90% of its hours recorded", "Common data-completeness rule; days below it are listed, not filled in"),
    ("Baseline", "12 months before (every season once)", "IPMVP / ASHRAE practice: a full annual cycle"),
    ("Expected-use model", "ASHRAE change-point model (temperature + weekday/holiday)", "ASHRAE Guideline 14 / Inverse Modeling Toolkit"),
    ("Balance temperatures", "found from the data", "every whole °C observed is tried; best by BIC"),
    ("Model accuracy grade", "Good ≤ 15% / OK ≤ 30% error; bias ≤ ±5% / ±10%", "ASHRAE Guideline 14 calibration limits (monthly / hourly data)"),
    ("Measurement window", "4, 6 or 8 whole weeks", "whole weeks = same weekday mix before and after"),
    ("Ghost twin", "the meter's better model (change-point or gradient boosting)", "picked per meter by its typical miss on 12 hidden months"),
    ("Gradient boosting", "LightGBM, library defaults", "no tuning — nothing to cherry-pick"),
    ("Placebo-in-time", "fake action every week of the year before", "same weekday mix for every fake date"),
    ("95% range", "t × typical fake saving × √(1+1/k)", "standard prediction interval; k = independent stretches in a year"),
    ("Significance", "p ≤ 5% (permutation test)", "statistical convention; VERIFIED needs range AND p"),
    ("Counted saving", "low end of the 95% range", "never overstate — the brief's requirement"),
    ("Alert bar", "biggest out-of-sample surprise of the baseline year", "each meter sets its own bar; no tuned thresholds"),
    ("Holidays", "Québec public holidays", "official calendar (python `holidays`)"),
    ("Electricity CO₂", "0.5 kg CO₂ / MWh", "Hydro-Québec, CO2 emissions and Hydro-Québec electricity 1990–2020 (2019 & 2020)"),
    ("Natural gas CO₂", "1.926 kg CO₂ / m³ — used only once the meter unit is confirmed",
     "Québec MELCC GHG quantification guide, Table 5"),
    ("Landfill CO₂e", "0.639 t CO₂e per tonne", "US EPA GHG Emission Factors Hub 2024: 0.58 t per short ton, mixed MSW landfilled"),
    ("E-waste CO₂e", "0.022 t CO₂e per tonne (end of life)", "US EPA GHG Emission Factors Hub 2024: 0.02 t per short ton, mixed electronics recycled"),
    ("E-waste routes", "keep / registered refurbisher / registered recycler", "India E-Waste (Management) Rules, 2022 (bulk consumer duties)"),
    ("E-waste expected life", "from the CPCB EPR portal, per EEE code", "entered with each device — never assumed"),
    ("Room vacancy switch-off", "20 minutes", "ASHRAE 90.1 occupancy-sensor rule (changeable in the app)"),
    ("Room comfort band", "80% acceptability, 7-day mean outdoor temperature", "ASHRAE 55 adaptive model via pythermalcomfort"),
    ("Fan credit", "0.6 m/s → +1.2 °C", "ASHRAE 55 elevated-air-speed step"),
    ("People counting", "YOLOv8n (COCO), default confidence", "Ultralytics; class 'person' only"),
    ("Waste diversion", "(compost + paper + mixed recycling) ÷ (those + landfill)", "bulky waste excluded: Concordia says its split is unknown"),
], columns=["What", "Value", "Why / source"])

_ok = {k: v for k, v in VERIFIED.items() if "saving_%" in v}
_ver = [v for v in _ok.values() if v["verdict"] == "VERIFIED SAVING"]
_gas = [v for v in _ok.values() if METER_INFO[v["point"]][1] == "gas"]
_top_e = RECS[(RECS.area == "Electricity") & RECS.priority.str.startswith("2")].head(1)
HEADLINE = []
HEADLINE.append(f"1. One real campus, every resource: {DAILY.shape[1]} meters (electricity, gas, water) + {len(WASTE)} waste invoices, "
                f"{HOURLY.index.min():%b %Y}–{HOURLY.index.max():%b %Y} — plus a live camera for room-level control.")
HEADLINE.append(f"2. It optimises: {len(RECS)} data-driven recommendations, each sized from the building's own data"
                + (f" — e.g. {_top_e.iloc[0]['where']}: {_top_e.iloc[0]['finding']} ({_top_e.iloc[0]['potential (not verified)']})." if len(_top_e) else "."))
HEADLINE.append(f"3. Trust test on {TRUST_SUMMARY['tests']} normal months (nothing done): a before/after dashboard showed a 'saving' "
                f"{TRUST_SUMMARY['naive_saving']} times; GreenLedger certified {TRUST_SUMMARY['gl_saving']}.")
if _gas:
    g = max(_gas, key=lambda v: v["naive_%"])
    HEADLINE.append(f"4. Not overstated: a before/after dashboard says gas fell {g['naive_%']:.0f}% after March 2020; the ghost twin "
                    f"(real weather, real calendar) says {g['saving_%']:.0f}% and the placebo test says that is normal noise "
                    f"(p = {g['p']:.2f}). Nothing counted.")
if len(DISCOVERY_TABLE) and _ver:
    first = DISCOVERY_TABLE.sort_values("first seen").iloc[0]
    best = max(_ver, key=lambda v: v["saving_%"])
    HEADLINE.append(f"5. Ground-truth test: nobody told GreenLedger about the COVID-19 closure — it flagged the drop on "
                    f"{pd.Timestamp(first['first seen']):%d %b %Y}, and the ghost twin measured {METER_INFO[best['point']][2]} "
                    f"electricity {best['saving_%']:.0f}% below normal (p = {best['p']:.3f}). The same engine verifies real actions.")
_cb = CARBON[CARBON.period == _period(BASE_START)]
HEADLINE.append(f"6. Carbon, honestly: in the baseline year landfill waste ≈ {_cb[_cb.scope.str.contains('landfill')]['t CO2e'].sum():,.0f} t CO₂e "
                f"vs metered electricity ≈ {_cb[_cb.scope.str.contains('electricity')]['t CO2e'].sum():,.0f} t (Québec hydro). Gas CO₂ is "
                f"withheld until the meter unit is confirmed; {len(FAULTS)} bad meter-days were excluded with reasons.")
print("\n".join(HEADLINE))


## Cell 12 · Export the report
HTML report + ZIP "labelled dataset" (daily readings with ghost-twin value, fault reason and alert id; alerts; actions; verification; recommendations & decisions; carbon; waste; e-waste; room log; sources). Files are in the 📁 panel under `greenledger_out`.

In [ ]:
# ---- Exportable report + labelled dataset (for the judges / facilities team) ----
def labelled_dataset():
    rows = []
    for p in DAILY:
        f = FAULTS[FAULTS.point == p].set_index("day").rule
        df = pd.DataFrame({"date": DAYS, "meter": METER_INFO[p][2], "resource": METER_INFO[p][1],
                           "system_name": METER_INFO[p][0], "actual": DAILY[p].values,
                           "expected": EXPECTED[p].reindex(DAYS).values, "temp_C": TEMP.values,
                           "off_day": OFF_DAY.values, "period": np.where(DAYS <= BASE_END, "baseline", "monitoring")})
        df["data_fault"] = df.date.map(f).fillna("")
        df["alert"] = ""
        for _, a in ALERT_TABLE[ALERT_TABLE.point == p].iterrows():
            df.loc[(df.date >= pd.Timestamp(a["from"])) & (df.date <= pd.Timestamp(a["to"])), "alert"] = a["id"]
        rows.append(df)
    return pd.concat(rows, ignore_index=True)


def verification_table():
    out = []
    for _, a in ACTIONS.iterrows():
        r = VERIFIED.get(a.action_id) or verify(a.point, a.date, 42, ACTIONS[ACTIONS.action_id != a.action_id])
        out.append({"action": a.action_id, "date": a.date.date(), "meter": METER_INFO[a.point][2], "title": a.title,
                    "verdict": r["verdict"], "change %": round(r.get("saving_%", np.nan), 1), "p": round(r.get("p", np.nan), 3),
                    "95% low": round(r.get("low_%", np.nan), 1), "95% high": round(r.get("high_%", np.nan), 1),
                    "naive %": round(r.get("naive_%", np.nan), 1), "counted": round(r.get("counted", 0), 0),
                    "unit": r["unit"], "explanation": explain(r)})
    return pd.DataFrame(out)


def build_report():
    stamp = datetime.datetime.now().strftime("%Y%m%d_%H%M")
    zpath = os.path.join(OUT_DIR, f"greenledger_concordia_{stamp}.zip")
    V = verification_table()
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("readings_daily_labelled.csv", labelled_dataset().to_csv(index=False))
        z.writestr("alerts.csv", ALERT_TABLE.drop(columns="point").to_csv(index=False))
        z.writestr("discoveries.csv", DISCOVERY_TABLE.drop(columns="point").to_csv(index=False))
        z.writestr("data_faults.csv", FAULTS.to_csv(index=False))
        z.writestr("actions.csv", ACTIONS.to_csv(index=False))
        z.writestr("verification.csv", V.to_csv(index=False))
        z.writestr("model_accuracy.csv", ACC.drop(columns="point").to_csv(index=False))
        z.writestr("trust_test.csv", TRUST.to_csv(index=False))
        z.writestr("waste_monthly_tonnes.csv", WASTE_MONTHLY.to_csv())
        z.writestr("waste_yearly.csv", WASTE_YEARLY.to_csv())
        z.writestr("parameters_and_sources.csv", PARAMETERS.to_csv(index=False))
        z.writestr("recommendations.csv", RECS.drop(columns=["_value", "_point"]).to_csv(index=False))
        z.writestr("recommendation_decisions.csv", DECISIONS.to_csv(index=False))
        z.writestr("carbon_inventory.csv", CARBON.to_csv(index=False))
        z.writestr("carbon_factors.csv", CARBON_FACTORS.to_csv(index=False))
        z.writestr("model_choice.csv", MODEL_CHECK.drop(columns="point").to_csv(index=False))
        z.writestr("room_control_log.csv", pd.DataFrame(ROOM["log"] if "ROOM" in globals() else []).to_csv(index=False))
        if EWASTE is not None:
            z.writestr("ewaste_plan.csv", EWASTE.to_csv(index=False))
        z.writestr("README.txt", "GreenLedger — reference campus: Concordia University (Montréal).\n"
                   "Source: Concordia University open data (Facilities), CC BY 4.0. Nothing simulated or filled in.\n"
                   "readings_daily_labelled.csv: one row per meter per day — actual, expected, data_fault reason, alert id.\n")
    tbl = lambda d: d.to_html(index=False, border=0, classes="t")
    html = f"""<html><head><meta charset="utf-8"><title>GreenLedger report</title><style>
body{{font-family:Inter,Arial,sans-serif;max-width:1000px;margin:30px auto;color:#1f2937;line-height:1.5}}
h1{{color:#047857}} .t{{border-collapse:collapse;width:100%;font-size:13px}} .t td,.t th{{border-bottom:1px solid #e5e7eb;padding:6px;text-align:left}}
.t th{{background:#f3f4f6}} li{{margin:6px 0}}</style></head><body>
<h1>GreenLedger — campus resource report</h1>
<p>Reference campus: <b>Concordia University, Montréal</b> · Concordia open data (CC BY 4.0) · generated {datetime.datetime.now():%d %b %Y %H:%M}</p>
<h2>Headlines</h2><ul>{''.join(f'<li>{h}</li>' for h in HEADLINE)}</ul>
<h2>Recommendations (potential — verified only after they are done)</h2>{tbl(RECS[['id', 'priority', 'area', 'where', 'finding', 'potential (not verified)', 'next step']])}
<h2>Carbon inventory (t CO₂e; blank = not measured / not computable)</h2>{CARBON_BY_PERIOD.reset_index().to_html(index=False, border=0, classes='t', na_rep='—')}
{tbl(CARBON_FACTORS)}
<h2>Did it work? (action register — ghost twin + placebo-in-time)</h2>{tbl(V[['action', 'date', 'meter', 'title', 'verdict', 'change %', '95% low', '95% high', 'p', 'naive %', 'counted', 'unit']])}
<h2>Alerts</h2>{tbl(ALERT_TABLE[['id', 'meter', 'from', 'to', 'extra use', 'what happened']])}
<h2>Discoveries (unexplained sustained drops)</h2>{tbl(DISCOVERY_TABLE.drop(columns='point'))}
<h2>Waste per 12-month period (tonnes)</h2>{WASTE_YEARLY.to_html(border=0, classes='t')}
<h2>Model accuracy</h2>{tbl(ACC.drop(columns='point'))}
<h2>Which model is the ghost twin?</h2>{tbl(MODEL_CHECK.drop(columns='point'))}
{'<h2>E-waste plan</h2>' + tbl(EWASTE[['device_id', 'device_type', 'department', 'age_years', 'expected_life_years', 'condition', 'route', 'why']]) if EWASTE is not None else ''}
{'<h2>Room control log</h2>' + tbl(pd.DataFrame(ROOM['log'])) if 'ROOM' in globals() and ROOM['log'] else ''}
<h2>Parameters and sources</h2>{tbl(PARAMETERS)}
</body></html>"""
    hpath = os.path.join(OUT_DIR, f"greenledger_report_{stamp}.html")
    open(hpath, "w").write(html)
    return hpath, zpath


REPORT_HTML, REPORT_ZIP = build_report()
print("Report:", REPORT_HTML)
print("Labelled dataset (ZIP):", REPORT_ZIP)


## Cell 13 · 🎥 Live room control engine
* **People** counted from the camera with **YOLOv8n**.
* **Comfort** from the **ASHRAE 55 adaptive model** (pythermalcomfort): the band 80% of people accept for this week's outdoor weather; a fan widens it by 1.2 °C.
* **Decision:** comfortable → nothing on · fan is enough → fan only · too warm → AC at the *warmest* comfortable setting · empty 20 min (**ASHRAE 90.1**) → all off.
* **Live weather** from Open-Meteo. **Switching** through a smart-switch webhook (`GL_DEVICE_WEBHOOK`); without one it runs in advice mode and logs what it would do.
* Energy avoided here is an **estimate**; the meter-based ghost twin turns it into a verified number.

The self-test runs on a real photo that ships with Ultralytics.

In [ ]:
# ---- 🎥 Live room control: camera → people count → AC / fan / lights decision ----
# • People are counted from the live camera with YOLOv8n (COCO-trained, default confidence) — no other sensors.
# • Comfort: ASHRAE Standard 55 adaptive comfort model (via `pythermalcomfort`). It gives the temperature band that
#   80% of people accept for the last 7 days' outdoor temperature, and how much a fan widens it
#   (0.6 m/s air speed → +1.2 °C, the standard's first elevated-air-speed step).
# • Vacancy: switch off when nobody has been seen for 20 minutes — ASHRAE 90.1 occupancy-sensor rule (changeable).
# • Live outdoor weather: Open-Meteo (free, no key). If offline, type the outdoor temperature yourself.
# • Switching: a real smart plug / Home Assistant can be driven through a webhook (set GL_DEVICE_WEBHOOK).
#   Without one, GreenLedger runs in "advice mode" and logs what it WOULD switch. Nothing is pretended.
# • Energy avoided here is an ESTIMATE (nameplate kW × minutes switched off). The meter-based ghost-twin test
#   (cell 8) is what turns it into a verified saving.
import time as _time, threading as _th, urllib.parse
try:
    from ultralytics import YOLO
    YOLO_PATH = os.path.join(OUT_DIR, "yolov8n.pt")         # downloaded once now (≈6 MB) → no network needed during the demo
    if not os.path.exists(YOLO_PATH):
        _m = YOLO("yolov8n.pt"); _src = getattr(_m, "ckpt_path", None) or "yolov8n.pt"
        import shutil; shutil.copy(_src, YOLO_PATH)
    YOLO_MODEL = YOLO(YOLO_PATH)
except Exception as e:
    YOLO_MODEL = None
    print("Camera detection unavailable:", e)
try:
    from pythermalcomfort.models import adaptive_ashrae
except Exception as e:
    adaptive_ashrae = None
    print("pythermalcomfort unavailable:", e)

VACANCY_MIN_DEFAULT = 20              # ASHRAE 90.1: occupancy sensors switch lighting off within 20 min of vacancy
FAN_AIR_SPEED = 0.6                   # m/s — ASHRAE 55 first elevated-air-speed step (+1.2 °C)
STILL_AIR = 0.1                       # m/s — still air in ASHRAE 55
BEE_DEFAULT_AC_C = 24                 # India BEE: default AC temperature setting 24 °C (for comparison)
DEVICE_WEBHOOK = os.environ.get("GL_DEVICE_WEBHOOK", "")
ROOM_LOG_CSV = os.path.join(OUT_DIR, "room_control_log.csv")
ROOM = {"last_person": None, "people": 0, "lock": _th.Lock(),
        "state": {"AC": {"on": False, "setpoint": None}, "Fan": {"on": False}, "Lights": {"on": False}},
        "since": {"AC": None, "Fan": None, "Lights": None}, "off_minutes": {"AC": 0.0, "Fan": 0.0, "Lights": 0.0},
        "log": [], "suggested": {}}
WEATHER = {"city": None, "lat": None, "lon": None, "now": None, "trm": None, "fetched": None, "error": None}


def fetch_weather(city):
    """Current outdoor temperature + 7-day mean (ASHRAE 55 prevailing mean outdoor temperature) from Open-Meteo."""
    try:
        g = json.loads(urllib.request.urlopen(
            "https://geocoding-api.open-meteo.com/v1/search?count=1&name=" + urllib.parse.quote(city), timeout=8).read())
        r0 = g["results"][0]
        w = json.loads(urllib.request.urlopen(
            f"https://api.open-meteo.com/v1/forecast?latitude={r0['latitude']}&longitude={r0['longitude']}"
            "&current=temperature_2m&daily=temperature_2m_mean&past_days=7&forecast_days=1&timezone=auto", timeout=8).read())
        past = [t for t in w["daily"]["temperature_2m_mean"][:7] if t is not None]
        WEATHER.update(city=f"{r0['name']}, {r0.get('country', '')}", lat=r0["latitude"], lon=r0["longitude"],
                       now=float(w["current"]["temperature_2m"]), trm=float(np.mean(past)),
                       fetched=datetime.datetime.now(), error=None)
    except Exception as e:
        WEATHER.update(error=str(e))
    return WEATHER


def comfort_band(t_running_mean, air_speed):
    """80%-acceptability band from ASHRAE 55 adaptive model (pythermalcomfort)."""
    r = adaptive_ashrae(tdb=25, tr=25, t_running_mean=t_running_mean, v=air_speed)
    get = (lambda k: getattr(r, k)) if not isinstance(r, dict) else (lambda k: r[k])
    return float(get("tmp_cmf_80_low")), float(get("tmp_cmf_80_up"))


def decide(people, vacant_min, room_temp, trm, vacancy_limit):
    """Return the wanted state of each device and one plain reason."""
    if people == 0 and vacant_min is not None and vacant_min >= vacancy_limit:
        return ({"AC": False, "Fan": False, "Lights": False, "setpoint": None},
                f"Room empty for {vacant_min:.0f} min (≥ {vacancy_limit} min) → everything off.")
    if people == 0:
        left = vacancy_limit - (vacant_min or 0)
        return (None, f"No one seen — waiting {left:.0f} more min before switching off (people may be out of view).")
    if room_temp is None or trm is None or adaptive_ashrae is None:
        return ({"AC": None, "Fan": None, "Lights": True, "setpoint": None},
                f"{people} people present. Enter the room temperature to get the AC/fan decision.")
    lo, hi = comfort_band(trm, STILL_AIR)
    _, hi_fan = comfort_band(trm, FAN_AIR_SPEED)
    if room_temp < lo:
        return ({"AC": False, "Fan": False, "Lights": True, "setpoint": None},
                f"{room_temp:.1f} °C is below the comfort band ({lo:.1f}–{hi:.1f} °C) → AC and fan off.")
    if room_temp <= hi:
        return ({"AC": False, "Fan": False, "Lights": True, "setpoint": None},
                f"{room_temp:.1f} °C is already comfortable ({lo:.1f}–{hi:.1f} °C for this week's weather) → no AC, no fan needed.")
    if room_temp <= hi_fan:
        return ({"AC": False, "Fan": True, "Lights": True, "setpoint": None},
                f"{room_temp:.1f} °C — a fan makes up to {hi_fan:.1f} °C comfortable (ASHRAE 55 air-speed credit) → fan on, AC off.")
    sp = float(np.floor(hi_fan))                              # whole °C, as on an AC remote, never above the band
    return ({"AC": True, "Fan": True, "Lights": True, "setpoint": sp},
            f"{room_temp:.1f} °C is too warm even with a fan → AC on at {sp:.1f} °C with the fan "
            f"(warmest comfortable setting; BEE default is {BEE_DEFAULT_AC_C} °C).")


def send_command(device, on, setpoint=None):
    """Drive a real switch if a webhook is configured; otherwise advice mode."""
    if not DEVICE_WEBHOOK:
        return "advice mode (no smart switch connected)"
    try:
        req = urllib.request.Request(DEVICE_WEBHOOK, method="POST", headers={"Content-Type": "application/json"},
                                     data=json.dumps({"device": device, "on": bool(on), "setpoint": setpoint}).encode())
        urllib.request.urlopen(req, timeout=5)
        return "sent to switch"
    except Exception as e:
        return f"switch error: {e}"


def apply_states(want, reason, auto, now):
    changes = []
    for dev in ("AC", "Fan", "Lights"):
        w = want.get(dev)
        if w is None:
            continue
        cur = ROOM["state"][dev]["on"]
        sp_change = dev == "AC" and w and want.get("setpoint") != ROOM["state"]["AC"].get("setpoint")
        if not auto:                                            # advice mode: log a suggestion only when it changes
            key = (w, want.get("setpoint") if dev == "AC" else None)
            if ROOM["suggested"].get(dev) == key or (w == cur and not sp_change):
                continue
            ROOM["suggested"][dev] = key
        if w != cur or sp_change:
            how = send_command(dev, w, want.get("setpoint") if dev == "AC" else None) if auto else "suggested — not switched (auto off)"
            if auto:
                if not w and cur:
                    ROOM["since"][dev] = now
                if w and ROOM["since"][dev] is not None:
                    ROOM["off_minutes"][dev] += (now - ROOM["since"][dev]).total_seconds() / 60
                    ROOM["since"][dev] = None
                ROOM["state"][dev]["on"] = w
                if dev == "AC":
                    ROOM["state"]["AC"]["setpoint"] = want.get("setpoint")
            changes.append({"time": now.strftime("%Y-%m-%d %H:%M:%S"), "device": dev,
                            "action": ("" if auto else "suggest ") + (("ON" + (f" @ {want.get('setpoint'):.0f} °C" if dev == "AC" and want.get("setpoint") else "")) if w else "OFF"),
                            "people": ROOM["people"], "reason": reason, "how": how})
    if changes:
        ROOM["log"].extend(changes)
        pd.DataFrame(ROOM["log"]).to_csv(ROOM_LOG_CSV, index=False)
    return changes


def process_frame(frame, room_temp=None, auto=False, vacancy_limit=VACANCY_MIN_DEFAULT, outdoor_manual=None):
    """One camera frame in → annotated frame, status, log out."""
    now = datetime.datetime.now()
    annotated, people = frame, 0
    if frame is not None and YOLO_MODEL is not None:
        bgr = np.ascontiguousarray(np.asarray(frame)[:, :, :3][:, :, ::-1])   # camera gives RGB; YOLO expects BGR
        r = YOLO_MODEL(bgr, classes=[0], verbose=False)[0]          # class 0 = person
        people = int(len(r.boxes))
        annotated = np.ascontiguousarray(r.plot()[:, :, ::-1])      # back to RGB for display
    with ROOM["lock"]:
        ROOM["people"] = people
        if people > 0:
            ROOM["last_person"] = now
        vacant = None if ROOM["last_person"] is None else (now - ROOM["last_person"]).total_seconds() / 60
        if ROOM["last_person"] is None and people == 0:
            vacant = float(vacancy_limit)                            # never seen anyone since start
        trm = WEATHER["trm"] if WEATHER["trm"] is not None else outdoor_manual
        want, reason = decide(people, vacant, room_temp, trm, int(vacancy_limit))
        changes = apply_states(want, reason, auto, now) if want else []
    return annotated, people, reason, changes


def estimated_kwh_avoided(nameplate_kw):
    now = datetime.datetime.now()
    tot = 0.0
    for dev, kw in nameplate_kw.items():
        if not kw:
            continue
        mins = ROOM["off_minutes"][dev] + ((now - ROOM["since"][dev]).total_seconds() / 60 if ROOM["since"][dev] else 0)
        tot += float(kw) * mins / 60
    return tot


print("Room control ready:", "camera detection ON (YOLOv8n)" if YOLO_MODEL else "camera detection OFF",
      "·", "comfort model ON (ASHRAE 55 adaptive)" if adaptive_ashrae else "comfort model OFF",
      "·", "switching via webhook" if DEVICE_WEBHOOK else "advice mode (no smart switch connected)")
# quick self-test on Ultralytics' own sample photo (a real street photo shipped with the library)
try:
    from ultralytics.utils import ASSETS
    import cv2
    _img = cv2.imread(str(ASSETS / "bus.jpg"))[:, :, ::-1]
    _a, _n, _why, _ = process_frame(_img, room_temp=None, auto=False)
    print(f"Self-test on Ultralytics sample photo: {_n} people detected.")
    ROOM.update(last_person=None, people=0, log=[], suggested={})            # start the real room with a clean log
except Exception as e:
    print("Self-test skipped:", e)


## Cell 14 · Build the app
Four tabs, in demo order: **Overview → Act & Prove → Live room → Data**. The what-if sliders on Overview are projections from each building's own model and last year's real weather — you choose the scenario, nothing is assumed.

In [ ]:
# ---- The easy app (Gradio). Everything on screen comes from the cells above. ----
import gradio as gr

CSS = """
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@500;700&family=Space+Grotesk:wght@600;700&display=swap');
:root{--bg:#0B1220;--s1:#111A2C;--s2:#182238;--s3:#1F2A44;--bd:#2A3A5C;--tx:#E6EDF7;--tx2:#B6C2D6;--mu:#8794AB;
--em:#10B981;--em2:#34D399;--am:#F59E0B;--am2:#FBBF24;--rd:#EF4444;--rd2:#F87171;--bl:#3B82F6;--bl2:#60A5FA;}
html,body,gradio-app,.gradio-container{background:var(--bg)!important;color:var(--tx)!important;font-family:Inter,system-ui,-apple-system,'Segoe UI',Roboto,sans-serif!important}
.gradio-container{max-width:1500px!important;margin:0 auto!important}
.gradio-container *{--body-background-fill:var(--bg);--background-fill-primary:var(--s1);--background-fill-secondary:var(--s2);
--block-background-fill:var(--s1);--block-border-color:var(--bd);--border-color-primary:var(--bd);--input-background-fill:var(--s2);
--body-text-color:var(--tx);--block-label-text-color:var(--tx2);--block-title-text-color:var(--tx2);--color-accent:var(--em);
--table-even-background-fill:var(--s1);--checkbox-label-background-fill:var(--s2);
--checkbox-label-background-fill-selected:rgba(16,185,129,.18);--checkbox-label-text-color:var(--tx);
--checkbox-label-text-color-selected:#fff;--checkbox-label-border-color:var(--bd);--block-label-background-fill:var(--s2);
--block-label-text-color:var(--tx2);--block-label-border-color:var(--bd);--input-placeholder-color:var(--mu);--table-odd-background-fill:var(--s2);--table-row-focus:var(--s3);--neutral-50:var(--s1);
--neutral-100:var(--s2);--neutral-200:var(--bd);--neutral-700:var(--tx);--neutral-800:var(--tx);--neutral-900:#fff}
.gl-head{display:flex;justify-content:space-between;align-items:center;gap:16px;flex-wrap:wrap;padding:22px 28px;margin:14px 0 10px;
background:linear-gradient(120deg,#111A2C,#16304a);border:1px solid var(--bd);border-radius:14px}
.gl-brand{display:flex;gap:14px;align-items:center}.gl-logo{width:50px;height:50px;border-radius:12px;background:linear-gradient(135deg,#10B981,#047857);
display:flex;align-items:center;justify-content:center;font:700 26px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:#fff}
.gl-title{font:700 26px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;color:#fff;letter-spacing:-.02em}.gl-sub{font:500 12px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);margin-top:4px;letter-spacing:.04em}
.gl-pill{font:700 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;padding:8px 12px;border-radius:8px;border:1px solid rgba(16,185,129,.45);background:rgba(16,185,129,.12);color:var(--em2)}
.gl-intro{background:var(--s1);border:1px solid var(--bd);border-left:4px solid var(--em);border-radius:10px;padding:14px 18px;margin:12px 0;font-size:14px;color:var(--tx2);line-height:1.6}
.gl-intro b{color:#fff}
.gl-cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(190px,1fr));gap:12px;margin:12px 0}
.gl-card{background:linear-gradient(160deg,var(--s1),var(--s2));border:1px solid var(--bd);border-left:4px solid var(--em);border-radius:12px;padding:14px 16px}
.gl-card.warn{border-left-color:var(--am)}.gl-card.bad{border-left-color:var(--rd)}.gl-card.info{border-left-color:var(--bl)}.gl-card.mute{border-left-color:#5E6B85}
.gl-v{font:700 26px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:#fff;letter-spacing:-.02em}.gl-u{font-size:13px;color:var(--tx2);margin-left:4px}
.gl-l{font:700 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);text-transform:uppercase;letter-spacing:.08em;margin-top:8px}
.gl-h{font-size:12px;color:var(--mu);margin-top:4px;line-height:1.45}
.gl-verdict{border-radius:14px;padding:20px 24px;margin:10px 0;border:1px solid var(--bd)}
.gl-verdict .big{font:700 30px 'Space Grotesk',Inter,system-ui,-apple-system,'Segoe UI',sans-serif;letter-spacing:-.01em}.gl-verdict .num{font:700 20px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin-top:6px;color:#fff}
.gl-verdict p{color:var(--tx2);font-size:14px;line-height:1.6;margin:10px 0 0}
.gl-ok{background:rgba(16,185,129,.12);border-color:rgba(16,185,129,.5)}.gl-ok .big{color:var(--em2)}
.gl-no{background:rgba(245,158,11,.10);border-color:rgba(245,158,11,.45)}.gl-no .big{color:var(--am2)}
.gl-bad{background:rgba(239,68,68,.10);border-color:rgba(239,68,68,.45)}.gl-bad .big{color:var(--rd2)}
.gl-wait{background:var(--s2)}.gl-wait .big{color:var(--tx2)}
table.gl-t{width:100%;border-collapse:collapse;font:13px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;margin:8px 0}
table.gl-t th{background:var(--s3);color:#fff;text-align:left;padding:9px 10px;font-size:11px;text-transform:uppercase;letter-spacing:.06em}
table.gl-t td{padding:8px 10px;border-bottom:1px solid var(--bd);color:var(--tx)}
.z-hi{background:rgba(239,68,68,.22)!important;color:#FECACA!important}.z-lo{background:rgba(16,185,129,.20)!important;color:#A7F3D0!important}
.gl-sec{font:700 12px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--tx2);letter-spacing:.12em;text-transform:uppercase;margin:22px 0 8px}
.gl-note{font-size:12.5px;color:var(--mu);font-style:italic;margin:6px 0}
.gl-msg{font:600 12.5px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--em2);padding:8px 12px;background:rgba(16,185,129,.1);border:1px solid rgba(16,185,129,.35);border-radius:8px;display:inline-block}
.gl-msg.warn{color:var(--am2);background:rgba(245,158,11,.1);border-color:rgba(245,158,11,.4)}
.tabs button{font:600 12.5px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace!important;letter-spacing:.06em!important;text-transform:uppercase!important}
.tabs button.selected{color:var(--em2)!important;border-color:var(--em2)!important}
button.primary{background:linear-gradient(135deg,#10B981,#059669)!important;color:#fff!important;border:none!important;font-weight:700!important}
button.secondary{background:var(--s2)!important;color:var(--tx)!important;border:1px solid var(--bd)!important;font-weight:600!important}
button.secondary:hover{border-color:var(--em2)!important;color:var(--em2)!important}
.gl-foot{margin:36px 0 18px;padding-top:14px;border-top:1px solid var(--bd);display:flex;justify-content:space-between;flex-wrap:wrap;gap:10px;
font:500 11px 'JetBrains Mono',ui-monospace,SFMono-Regular,Menlo,Consolas,monospace;color:var(--mu);letter-spacing:.06em;text-transform:uppercase}
.gl-steps{display:grid;grid-template-columns:repeat(auto-fit,minmax(240px,1fr));gap:12px;margin:10px 0 4px}
.gl-step{background:var(--s1);border:1px solid var(--bd);border-radius:12px;padding:14px 16px}
.gl-step .n{display:inline-flex;width:28px;height:28px;border-radius:50%;background:rgba(16,185,129,.18);color:var(--em2);
align-items:center;justify-content:center;font-weight:700;margin-right:8px}
.gl-step b{color:#fff;font-size:16px}.gl-step p{color:var(--tx2);font-size:14px;margin:8px 0 0;line-height:1.5}
.gl-big{font-size:15px;color:var(--tx);line-height:1.65}
.gl-alert{background:var(--s1);border:1px solid var(--bd);border-left:4px solid var(--rd);border-radius:10px;padding:12px 16px;margin:8px 0;font-size:14.5px;line-height:1.55;color:var(--tx)}
.gl-alert.good{border-left-color:var(--em)} .gl-alert small{color:var(--mu);display:block;margin-top:4px}
.tabs button{font:600 14px Inter,system-ui,sans-serif!important;text-transform:none!important;letter-spacing:0!important}
</style>
"""
PL = dict(plot_bgcolor="#111A2C", paper_bgcolor="#111A2C", font=dict(family="Inter, sans-serif", size=13, color="#E6EDF7"),
          margin=dict(t=56, b=40, l=60, r=20), legend=dict(orientation="h", y=-0.15, bgcolor="rgba(0,0,0,0)"),
          hoverlabel=dict(bgcolor="#1F2A44", font=dict(color="#fff")))
GRID = dict(gridcolor="#243352", zerolinecolor="#324369", linecolor="#324369")


def fig_(f, title, h=380):
    f.update_layout(title=dict(text=title, font=dict(size=16, color="#fff")), height=h, **PL)
    f.update_xaxes(**GRID); f.update_yaxes(**GRID)
    return f


def blank(text):
    f = go.Figure(); f.add_annotation(text=text, x=.5, y=.5, xref="paper", yref="paper", showarrow=False,
                                       font=dict(color="#8794AB", size=14))
    f.update_layout(height=160, **PL); f.update_xaxes(visible=False); f.update_yaxes(visible=False)
    return f


def card(v, label, help_="", cls=""):
    h = f'<div class="gl-h">{help_}</div>' if help_ else ""
    return f'<div class="gl-card {cls}"><div class="gl-v">{v}</div><div class="gl-l">{label}</div>{h}</div>'


def tbl(df):
    return df.to_html(index=False, border=0, classes="gl-t", escape=False)


def meter_name(p):
    return f"{RESOURCE[METER_INFO[p][1]]['icon']} {RESOURCE[METER_INFO[p][1]]['label']} · {METER_INFO[p][2]}"


METER_CHOICES = [(meter_name(p), p) for p in DAILY]
EVENT_DATES = sorted(set(ACTIONS.date))
CAMPUS = CAMPUS_TOTAL
TYPICAL_MISS = TRUST.groupby("point")["gl_%"].apply(lambda s: float(np.sqrt((s ** 2).mean()))).to_dict()
# a month is offered if the campus electricity meters have ≥ 20 good days in it
MONTHS = [m for m in pd.period_range(DAYS[0], DAYS[-1], freq="M")
          if DAILY.loc[m.start_time:m.end_time.normalize(), CAMPUS["electricity"]].notna().sum().min() >= 20]
MONTH_CHOICES = [(m.strftime("%B %Y"), str(m)) for m in MONTHS]
PERIOD_LABEL = f"{BASE_START:%b}→{(BASE_START - pd.Timedelta(days=1)):%b}"


def verdict_style(v):
    return {"VERIFIED SAVING": ("gl-ok", "✅"), "VERIFIED INCREASE": ("gl-bad", "⛔"), "NOT PROVEN": ("gl-no", "🤔")}.get(v, ("gl-wait", "⏳"))


def rec_cards(df):
    decided = DECISIONS.drop_duplicates("rec_id", keep="last").set_index("rec_id")["decision"].to_dict() if len(DECISIONS) else {}
    def badge(rid):
        d = decided.get(rid)
        if not d:
            return ' <span class="gl-msg warn" style="font-size:11px;padding:2px 8px">⏳ waiting for a decision</span>'
        return (' <span class="gl-msg" style="font-size:11px;padding:2px 8px">✔ approved</span>' if d == "approved"
                else ' <span class="gl-msg warn" style="font-size:11px;padding:2px 8px">✋ rejected</span>')
    edge = {"1": "#EF4444", "2": "#F59E0B", "3": "#3B82F6"}
    return "".join(f'<div class="gl-alert" style="border-left-color:{edge[r.priority[0]]}"><b>{r.id} · {r.priority[4:]} · {r.area} · '
                   f'{r["where"]}</b>{badge(r.id)}<br>{r.finding}<small>💡 {r["next step"]} &nbsp;·&nbsp; potential (not verified): '
                   f'{r["potential (not verified)"]}</small></div>' for _, r in df.iterrows())


def top_recs():
    done = set(DECISIONS.rec_id) if len(DECISIONS) else set()
    df = RECS[~RECS.id.isin(done)]
    return pd.concat([df[df.priority.str.startswith("1")].head(1),
                      df[df.priority.str.startswith("2") & (df.area != "Waste")].head(1),
                      df[df.priority.str.startswith("3") & (df.area == "Electricity")].head(1)])


# ---------------- OVERVIEW ----------------
def ui_overview(month):
    mo = pd.Period(month, "M"); a, b = mo.start_time, mo.end_time.normalize()
    cards = []
    for res, pts in CAMPUS.items():
        act = sum(float(DAILY[p].loc[a:b].sum()) for p in pts)
        ok = all(p in RELIABLE for p in pts)
        exp_ = sum(float(EXPECTED[p].reindex(DAILY[p].loc[a:b].dropna().index).sum()) for p in pts)
        unit = RESOURCE[res]["unit"]; val = f"{act / 1000:,.0f} MWh" if unit == "kWh" else f"{act:,.0f} {unit}"
        if ok and exp_ > 0:
            d = (act - exp_) / exp_ * 100
            band = max(TYPICAL_MISS.get(p, 0) for p in pts)
            if abs(d) <= band:
                cls, txt = "", f"🟢 normal — {d:+.0f}% vs ghost twin (normal range ±{band:.0f}%)"
            elif d < 0:
                cls, txt = "info", f"🔵 {abs(d):.0f}% BELOW the ghost twin — beyond normal (±{band:.0f}%)"
            else:
                cls, txt = "bad", f"🔴 {d:.0f}% ABOVE the ghost twin — beyond normal (±{band:.0f}%)"
        else:
            cls, txt = "mute", "no reliable expectation for this meter — raw total only"
        cards.append(card(val, f"{RESOURCE[res]['icon']} {RESOURCE[res]['label']} · campus", txt, cls))
    wt = WASTE[(WASTE.Date >= a) & (WASTE.Date <= b)]
    div = DIVERSION.get(a, np.nan)
    cards.append(card(f"{wt.tonnes.sum():,.1f} t", "🗑️ Waste · invoiced",
                      f"{len(wt)} invoices" + (f" · diversion {div:.0f}%" if div == div else ""), "warn"))
    e_kwh = sum(float(DAILY[p].loc[a:b].sum()) for p in CAMPUS["electricity"])
    l_t = float(wt[wt.Waste_Type == "Landfill"].tonnes.sum())
    cards.append(card(f"{(e_kwh * ELEC_KG_CO2_PER_KWH / 1000 + l_t * LANDFILL_T_CO2E_PER_T):,.1f} t", "🌍 CO₂e measured",
                      "electricity + landfill (gas pending unit)", "mute"))
    al = ALERT_TABLE[(pd.to_datetime(ALERT_TABLE["to"]) >= a) & (pd.to_datetime(ALERT_TABLE["from"]) <= b)]
    cards.append(card(str(len(al)), "🚨 Alerts this month", "see Alerts", "bad" if len(al) else ""))
    return '<div class="gl-cards">' + "".join(cards) + "</div>"


def ui_meter_chart(p):
    y, e = DAILY[p], EXPECTED[p].reindex(DAYS)
    f = go.Figure()
    f.add_trace(go.Scatter(x=DAYS, y=e, name="Ghost twin (expected for that weather & day)", line=dict(color="#CBD5E1", dash="dot", width=1.5)))
    f.add_trace(go.Scatter(x=DAYS, y=y, name="Actual", line=dict(color=RESOURCE[METER_INFO[p][1]]["color"], width=2)))
    f.add_vrect(x0=BASE_START, x1=BASE_END, fillcolor="rgba(148,163,184,.06)", line_width=0,
                annotation_text="baseline year (learning)", annotation_position="top left", annotation_font_color="#8794AB")
    for _, r in ALERT_TABLE[ALERT_TABLE.point == p].iterrows():
        f.add_vrect(x0=pd.Timestamp(r["from"]), x1=pd.Timestamp(r["to"]) + pd.Timedelta(days=1), fillcolor="rgba(239,68,68,.25)", line_width=0)
    for _, r in DISCOVERY_TABLE[DISCOVERY_TABLE.point == p].iterrows():
        f.add_vrect(x0=pd.Timestamp(r["from"]), x1=pd.Timestamp(r["to"]) + pd.Timedelta(days=1), fillcolor="rgba(16,185,129,.15)", line_width=0)
    for d in EVENT_DATES:
        f.add_vline(x=d, line=dict(color="#FBBF24", dash="dash"))
    note = "" if p in RELIABLE else " — model NOT accurate enough for savings claims"
    fig_(f, f"{meter_name(p)} — daily use ({RESOURCE[METER_INFO[p][1]]['unit']}){note}", 420)
    f.add_annotation(text="red = alert · green = unexplained drop · yellow line = logged event", xref="paper", yref="paper",
                     x=1, y=1.08, showarrow=False, font=dict(size=11, color="#8794AB"), xanchor="right")
    return f


def ui_map():
    B = BUILDINGS.copy()
    B["code"] = B["Building"].astype(str)
    covered = {}
    for p, v in METER_INFO.items():
        codes = B[B.Campus == "LOY"].code.tolist() if v[3] == "LOY campus" else [c for c in v[3].split(",") if c]
        for c in codes:
            covered.setdefault(c, []).append(f"{RESOURCE[v[1]]['icon']} {v[2]}")
    wt = WASTE.groupby("Building").tonnes.sum()
    B["waste_t"] = B.code.map(lambda c: float(wt.get(c, wt.get("GN", 0) if c == "GNA" else 0)))   # invoices say "GN" for Grey Nuns
    B["meters"] = B.code.map(lambda c: "<br>".join(covered.get(c, [])) or "no meter in the open data")
    B["metered"] = B.code.map(lambda c: c in covered)
    B["hover"] = ("<b>" + B["Building Long Name"] + "</b> (" + B.code + ")<br>" + B.Address + "<br>" + B.meters +
                  B.waste_t.map(lambda t: f"<br>🗑️ {t:,.0f} t waste invoiced" if t else ""))
    f = go.Figure()
    kind = "map" if hasattr(go, "Scattermap") else ("mapbox" if hasattr(go, "Scattermapbox") else "geo")
    Scatter = {"map": getattr(go, "Scattermap", None), "mapbox": getattr(go, "Scattermapbox", None), "geo": go.Scattergeo}[kind]
    for metered, col, nm in [(False, "#64748B", "No meter in open data"), (True, "#10B981", "Metered building")]:
        s = B[B.metered == metered]
        f.add_trace(Scatter(lat=s.Latitude, lon=s.Longitude, mode="markers", name=nm, text=s.hover, hoverinfo="text",
                            marker=dict(size=(8 + np.sqrt(s.waste_t) * 1.2).tolist(), color=col, opacity=.85)))
    center = dict(lat=float(B.Latitude.mean()), lon=float(B.Longitude.mean()))
    if kind == "geo":
        f.update_geos(fitbounds="locations", bgcolor="#111A2C", showland=True, landcolor="#1F2A44")
    else:
        f.update_layout(**{kind: dict(style="carto-darkmatter" if kind == "map" else "open-street-map", center=center, zoom=12)})
    f.update_layout(height=430, margin=dict(t=40, b=0, l=0, r=0), paper_bgcolor="#111A2C", font=dict(color="#E6EDF7"),
                    legend=dict(orientation="h", y=0, bgcolor="rgba(17,26,44,.8)"),
                    title=dict(text=f"Concordia's {len(B)} buildings — both campuses (dot size = waste invoiced)", font=dict(size=15, color="#fff")))
    return f


# ---------------- RECOMMENDATIONS (human in the loop) ----------------
REC_CHOICES = [(f"{r.id} · {r.area} · {r['where']}", r.id) for _, r in RECS.iterrows()]


def decisions_html():
    return tbl(DECISIONS) if len(DECISIONS) else '<div class="gl-note">No decisions yet.</div>'


def ui_rec_detail(rid):
    if not rid:
        return ""
    return rec_cards(RECS[RECS.id == rid])


def ui_rec_decide(rid, decision, by, reason, date):
    keep = (decisions_html(), gr.update(), gr.update(), gr.update(), gr.update())
    if not rid:
        return ('<span class="gl-msg warn">⚠ Pick a recommendation</span>',) + keep
    if decision == "rejected" and not str(reason).strip():
        return ('<span class="gl-msg warn">⚠ Give a reason for rejecting — it is kept in the log</span>',) + keep
    d = None
    if decision == "approved" and str(date).strip():
        try:
            d = pd.Timestamp(str(date).strip())
        except Exception:
            return ('<span class="gl-msg warn">⚠ Date must be YYYY-MM-DD</span>',) + keep
    aid = decide_rec(rid, decision, by, reason, d)
    if aid:
        VERIFIED[aid] = verify(ACTIONS.set_index("action_id").loc[aid, "point"], d, 42, ACTIONS[ACTIONS.action_id != aid])
    is_waste = not isinstance(RECS.set_index("id").loc[rid, "_point"], str)
    msg = (f"✔ {rid} approved → logged as {aid} in the action register below; ghost-twin result: {VERIFIED[aid]['verdict']} "
           f"(it needs 4–8 weeks of meter data after the date)" if aid else
           f"✔ {rid} {decision}" + (" — waste actions have no meter; track them through the diversion rate in Data." if is_waste and decision == "approved" else ""))
    return (f'<span class="gl-msg">{msg}</span>', decisions_html(), gr.update(choices=action_choices(), value=aid or None),
            rec_cards(RECS), register_html(), rec_cards(top_recs()))


# ---------------- ALERTS ----------------
def ui_alerts():
    if not len(ALERT_TABLE):
        html = '<div class="gl-alert good">No unusual increases after the baseline year.</div>'
    else:
        html = "".join(f'<div class="gl-alert"><b>{r.id} · {r.meter}</b> · {r["from"]:%d %b %Y}'
                       f'{"" if r["from"] == r["to"] else " → " + r["to"].strftime("%d %b %Y")}<br>{r["what happened"]}'
                       f'<small>Extra use: {r["extra use"]} ({r["extra %"]}% above the ghost twin)</small></div>'
                       for _, r in ALERT_TABLE.iterrows())
    html += ('<div class="gl-sec">Discoveries — sustained drops nobody has explained yet</div>'
             '<div class="gl-note">Not counted as savings. If you know what caused one, log it as an action and verify it.</div>')
    html += "".join(f'<div class="gl-alert good"><b>{r.meter}</b> · {r["from"]:%d %b} → {r["to"]:%d %b %Y}<br>'
                    f'Used {r["less than expected"]} ({r["%"]}%) less than the ghost twin — more than any week of the baseline year.'
                    f'<small>First flagged {r["first seen"]:%a %d %b %Y}</small></div>' for _, r in DISCOVERY_TABLE.iterrows())
    return html


def ui_alert_chart(aid):
    if not aid:
        return blank("Pick an alert")
    r = ALERT_TABLE.set_index("id").loc[aid]
    p = r.point; a = pd.Timestamp(r["from"]) - pd.Timedelta(days=21); b = pd.Timestamp(r["to"]) + pd.Timedelta(days=14)
    y, e = DAILY[p].loc[a:b], EXPECTED[p].reindex(DAYS).loc[a:b]
    f = go.Figure([go.Bar(x=y.index, y=y, name="Actual", marker_color=[
        "#EF4444" if pd.Timestamp(r["from"]) <= d <= pd.Timestamp(r["to"]) else "#334155" for d in y.index]),
        go.Scatter(x=e.index, y=e, name="Ghost twin", line=dict(color="#FBBF24", width=2))])
    return fig_(f, f"{aid} · {meter_name(p)}", 380)


# ---------------- PROVE IT (ghost twin + placebo-in-time) ----------------
def action_choices():
    return [(f"{a.action_id} · {a.date:%d %b %Y} · {METER_INFO[a.point][2]} — {a.title[:50]}", a.action_id) for _, a in ACTIONS.iterrows()]


def register_html():
    rows = []
    for _, a in ACTIONS.iterrows():
        r = VERIFIED.get(a.action_id)
        v = r["verdict"] if r else "—"
        col = "#34D399" if "SAVING" in v else ("#F87171" if "INCREASE" in v else "#FBBF24")
        rows.append({"ID": a.action_id, "Date": f"{a.date:%d %b %Y}", "Meter": meter_name(a.point), "What": a.title,
                     "Type": a.type, "Result (6 weeks)": f'<b style="color:{col}">{v}</b>'})
    return tbl(pd.DataFrame(rows))


def ui_verify(aid, window):
    e0 = blank("")
    if not aid:
        return '<div class="gl-verdict gl-wait"><div class="big">Pick an action</div></div>', e0, e0, ""
    a = ACTIONS.set_index("action_id").loc[aid]
    r = verify(a.point, a.date, int(window), ACTIONS[ACTIONS.action_id != aid])
    if int(window) == 42:
        VERIFIED[aid] = r
    cls, icon = verdict_style(r["verdict"])
    head = f'<div class="gl-verdict {cls}"><div class="big">{icon} {r["verdict"]}</div>'
    if "saving_%" not in r:
        return head + f'<p>{r["why"]}</p></div>', blank(r["why"]), e0, ""
    head += (f'<div class="num">{abs(r["saving_%"]):.1f}% {"less" if r["saving_%"] > 0 else "more"} than the ghost twin · '
             f'95% range {r["low_%"]:.1f}% … {r["high_%"]:.1f}% · p = {r["p"]:.3f}</div>'
             f'<p class="gl-big">{explain(r)}</p></div>')
    u = r["unit"]
    cards = '<div class="gl-cards">' + "".join([
        card(f"{r['naive_%']:.0f}%", "Simple before/after says", "what most dashboards would report", "warn"),
        card(f"{r['saving_%']:.0f}%", "Ghost twin says", f"95% range {r['low_%']:.0f}% to {r['high_%']:.0f}%", "info"),
        card(f"{r['p']:.3f}", "Placebo p-value", f"{len(r['fakes'])} fake action dates last year", "mute"),
        card(f"{r['counted']:,.0f} {u}" if r["counted"] else "0", "Counted saving", "low end of the range — or zero", ""),
    ]) + "</div>"
    T0 = r["date"]; pre = DAILY[a.point].loc[T0 - pd.Timedelta(days=56): T0 - pd.Timedelta(days=1)]
    e, y = r["series_exp"], r["series_act"]
    half = r["high_%"] - r["saving_%"]                                 # band = ghost twin ± the placebo range
    f = go.Figure()
    f.add_trace(go.Scatter(x=e.index, y=e * (1 + half / 100), line=dict(width=0), showlegend=False, hoverinfo="skip"))
    f.add_trace(go.Scatter(x=e.index, y=e * (1 - half / 100), fill="tonexty", fillcolor="rgba(203,213,225,.15)", line=dict(width=0),
                           name="Ghost twin ± placebo range (95%)", hoverinfo="skip"))
    f.add_trace(go.Scatter(x=e.index, y=e, name="Ghost twin (no action, real weather & calendar)", line=dict(color="#CBD5E1", dash="dash")))
    f.add_trace(go.Scatter(x=pre.index, y=pre, name="Actual before", line=dict(color="#64748B")))
    f.add_trace(go.Scatter(x=y.index, y=y, name="Actual after", line=dict(color=RESOURCE[METER_INFO[a.point][1]]["color"], width=3)))
    f.add_vline(x=T0, line=dict(color="#FBBF24", dash="dot"))
    f = fig_(f, f"{meter_name(a.point)} — {a.title[:60]}", 420)
    h = go.Figure(go.Histogram(x=r["fakes"], nbinsx=20, marker_color="#3B82F6", opacity=.8, name="Fake action dates"))
    h.add_vline(x=r["saving_%"], line=dict(color="#34D399", width=3), annotation_text=f"real {r['saving_%']:.1f}%",
                annotation_font_color="#34D399")
    lo_x = min(min(r["fakes"]), r["saving_%"]) - 5; hi_x = max(max(r["fakes"]), r["saving_%"]) + 5
    if lo_x <= r["naive_%"] <= hi_x:
        h.add_vline(x=r["naive_%"], line=dict(color="#FBBF24", dash="dash"), annotation_text=f"naive {r['naive_%']:.0f}%",
                    annotation_position="bottom right", annotation_font_color="#FBBF24")
    h.update_xaxes(range=[lo_x, hi_x])
    h.add_vline(x=0, line=dict(color="#8794AB", dash="dot"))
    h = fig_(h, "Placebo-in-time: 'savings' found on fake dates when nothing was done", 420)
    h.update_xaxes(title="apparent saving %"); h.update_yaxes(title="fake dates")
    return head, f, h, cards


def ui_add(point, date, title, claimed):
    try:
        d = pd.Timestamp(str(date).strip())
        assert title and str(title).strip(), "Describe the action"
        assert point in DAILY, "Pick a meter"
    except Exception as ex:
        return f'<span class="gl-msg warn">⚠ {ex}</span>', gr.update(), register_html()
    aid = add_action(d, point, str(title).strip(), _num(claimed))
    VERIFIED[aid] = verify(point, d, 42, ACTIONS[ACTIONS.action_id != aid])
    return (f'<span class="gl-msg">✔ Logged {aid} — result: {VERIFIED[aid]["verdict"]}</span>',
            gr.update(choices=action_choices(), value=aid), register_html())


def ui_trust_chart():
    S = TRUST_SUMMARY; n = max(S["tests"], 1)
    f = go.Figure(go.Bar(x=["Simple before/after dashboard", "GreenLedger"], y=[S["naive_saving"] / n * 100, S["gl_saving"] / n * 100],
                         marker_color=["#FBBF24", "#34D399"], text=[f"{S['naive_saving']} of {n}", f"{S['gl_saving']} of {n}"],
                         textposition="outside", textfont=dict(size=18, color="#fff")))
    f = fig_(f, f"{n} normal months where NOTHING was done — how often was a 'saving' reported?", 400)
    f.update_yaxes(title="% of months", range=[0, max(S["naive_saving"] / n * 100, 10) * 1.3])
    return f


# ---------------- LIVE ROOM ----------------
def room_status_html(people, reason):
    w = WEATHER
    wtxt = (f"🌡️ Outdoor {w['now']:.1f} °C now · 7-day mean {w['trm']:.1f} °C ({w['city']}, Open-Meteo)" if w["trm"] is not None
            else f"🌡️ Live weather not loaded{(' — ' + w['error'][:80]) if w['error'] else ''}. Type the outdoor 7-day mean.")
    names = {"AC": "❄️ AC", "Fan": "🌀 Fan", "Lights": "💡 Lights"}
    dev = ""
    for d in ("AC", "Fan", "Lights"):
        on = ROOM["state"][d]["on"]
        txt = "🟢 ON" if on else "⚫ OFF"
        sp = ROOM["state"]["AC"].get("setpoint")
        if d == "AC" and on and sp:
            txt += f" · {sp:.0f} °C"
        dev += card(txt, names[d], "", "" if on else "mute")
    return (f'<div class="gl-cards">{card(str(people), "👥 People in view", "YOLOv8n live count", "info")}{dev}</div>'
            f'<div class="gl-alert good"><b>Decision:</b> {reason}<small>{wtxt} · '
            f'{"switching via smart switch" if DEVICE_WEBHOOK else "advice mode — no smart switch connected"}</small></div>')


def _num(x):
    try:
        return float(str(x).strip()) if str(x).strip() not in ("", "None") else None
    except ValueError:
        return None


def ui_room(frame, room_temp, auto, vacancy, outdoor_manual, kw_ac, kw_fan, kw_l):
    kw_ac, kw_fan, kw_l = _num(kw_ac), _num(kw_fan), _num(kw_l)
    if frame is None:
        return None, room_status_html(ROOM["people"], "Waiting for the camera…"), ui_room_log(), ""
    rt, om = _num(room_temp), _num(outdoor_manual)
    img, n, reason, _ = process_frame(frame, rt, bool(auto), int(vacancy or VACANCY_MIN_DEFAULT), om)
    kwh = estimated_kwh_avoided({"AC": kw_ac, "Fan": kw_fan, "Lights": kw_l})
    est = (f'<span class="gl-msg">≈ {kwh:,.2f} kWh avoided so far (estimate: nameplate kW × minutes switched off — '
           f'verify with the meter in “Prove it”)</span>' if any([kw_ac, kw_fan, kw_l]) else
           '<span class="gl-note">Enter each device\'s nameplate kW (from its label) to estimate energy avoided.</span>')
    return img, room_status_html(n, reason), ui_room_log(), est


def ui_room_log():
    return pd.DataFrame(ROOM["log"][-50:][::-1], columns=["time", "device", "action", "people", "reason", "how"])


def ui_weather(city):
    fetch_weather(city)
    return room_status_html(ROOM["people"], "Weather updated.")


def ui_manual(dev, on):
    now = datetime.datetime.now()
    want = {d: (on if d == dev else None) for d in ("AC", "Fan", "Lights")}
    want["setpoint"] = ROOM["state"]["AC"]["setpoint"]
    apply_states(want,
                 f"manual {dev} {'ON' if on else 'OFF'}", True, now)
    return room_status_html(ROOM["people"], f"Manual: {dev} switched {'ON' if on else 'OFF'}."), ui_room_log()


# ---------------- WASTE + E-WASTE ----------------
def ui_waste_monthly():
    M = WASTE_MONTHLY.copy()
    f = go.Figure([go.Bar(x=M.index, y=M[c], name=STREAM_LABEL.get(c, c), marker_color=STREAM_COLOR.get(c, "#888"))
                   for c in ["Landfill", "Organics", "Paper", "Mixed_recycling", "Bulky_waste"] if c in M])
    f.update_layout(barmode="stack")
    for d in EVENT_DATES:
        f.add_vline(x=d, line=dict(color="#FBBF24", dash="dash"))
    f = fig_(f, "Waste collected per month (tonnes, from invoices)", 400)
    f.update_yaxes(title="tonnes")
    return f


def ui_waste_buildings():
    W = WASTE_BY_BUILDING.drop(columns="Total t")
    f = go.Figure([go.Bar(y=W.index, x=W[c], orientation="h", name=STREAM_LABEL.get(c, c), marker_color=STREAM_COLOR.get(c, "#888"))
                   for c in W.columns])
    f.update_layout(barmode="stack")
    return fig_(f, "Where the bins are (tonnes, all years)", 340)


def ui_waste_cards():
    full = WASTE_YEARLY.dropna(subset=["Diversion %"])
    cs = [card(f"{r['Diversion %']:.0f}%", f"Diversion {i}", f"{r['Invoices from']} → {r['to']}",
               "" if r["Diversion %"] >= full["Diversion %"].iloc[0] else "warn") for i, r in full.iterrows()]
    cs.append(card(f"{LANDFILL_BASE_CO2E:,.0f} t", "Landfill CO₂e (baseline yr)", f"{LANDFILL_BASE_T:,.0f} t × EPA factor", "bad"))
    cs.append(card(f"${WASTE_COST_PER_T.get('Landfill', np.nan):,.0f}", "Landfill cost / tonne", "as invoiced", "bad"))
    cs.append(card(str(NO_WEIGHT), "Pickups with no weight", "left out — never guessed", "mute"))
    return '<div class="gl-cards">' + "".join(cs) + "</div>"


def ewaste_view(df):
    if df is None or not len(df):
        return (f'<div class="gl-alert good">No asset register loaded. Concordia\'s open data has no e-waste records, so nothing is shown '
                f'rather than invented. Download the template, list your department\'s real devices (expected life from the CPCB EPR portal '
                f'for each EEE code), and upload it.</div>'), blank("No register loaded")
    leave = df[df["leaving campus"]]
    cs = '<div class="gl-cards">' + "".join([
        card(str(len(df)), "Devices in register", "", "info"),
        card(str(int((df.age_years >= df.expected_life_years).sum())), "Past expected life", "CPCB average life", "warn"),
        card(str(len(leave)), "To hand over", "registered refurbisher / recycler", "bad"),
        card(f"{leave.weight_kg.sum():,.0f} kg" if leave.weight_kg.notna().any() else "—", "Weight leaving campus", "where recorded", "mute"),
        card(f"{df.loc[df.route == 'Registered refurbisher', 'embodied_kgco2e'].sum():,.0f} kg" if df.embodied_kgco2e.notna().any() else "—",
             "Embodied CO₂e kept in use", "from manufacturer PCF sheets", ""),
    ]) + "</div>"
    g = df.groupby(["department", "route"]).size().unstack(fill_value=0)
    f = go.Figure([go.Bar(x=g.index, y=g[c], name=c) for c in g.columns]); f.update_layout(barmode="stack")
    return cs + tbl(df[["device_id", "device_type", "eee_code", "department", "location", "age_years", "expected_life_years",
                        "life_used_%", "condition", "route", "why"]]), fig_(f, "E-waste routes by department", 360)


def ui_ewaste_upload(file):
    global EWASTE
    if file is None:
        return ewaste_view(EWASTE)
    path = file if isinstance(file, str) else file.name
    try:
        raw = pd.read_csv(path)
        missing = [c for c in ["device_id", "device_type", "purchase_date", "expected_life_years", "condition"] if c not in raw]
        if missing:
            return f'<span class="gl-msg warn">⚠ Missing columns: {missing}</span>', blank("")
        EWASTE = ewaste_plan(raw)
        EWASTE.to_csv(os.path.join(OUT_DIR, "ewaste_plan.csv"), index=False)
    except Exception as ex:
        return f'<span class="gl-msg warn">⚠ {ex}</span>', blank("")
    return ewaste_view(EWASTE)


# ---------------- CARBON ----------------
def ui_carbon_html():
    _cb = CARBON[CARBON.period == _period(BASE_START)]
    e = float(_cb[_cb.scope.str.contains("electricity")].amount.sum())
    av = AVOIDED.copy()
    av["kg CO2e avoided"] = av["kg CO2e avoided"].round(0)
    av["counted"] = av["counted"].map(lambda v: f"{v:,.0f}")
    return ('<div class="gl-cards">'
            + card(f"{_cb['t CO2e'].sum():,.0f} t", f"CO₂e measured · {_period(BASE_START)}", "electricity + landfill", "info")
            + card(f"{e / 1e6:,.1f} GWh", "Electricity metered", f"= {e * ELEC_KG_CO2_PER_KWH / 1000:,.0f} t CO₂ on Québec hydro", "")
            + card(f"{e * INDIA_GRID_KG_PER_KWH / 1000:,.0f} t", "Same kWh on India's grid", f"CEA {INDIA_GRID_KG_PER_KWH} kg/kWh", "warn")
            + card(f"{GAS_FT3_T:,.0f}–{GAS_DOC_T:,.0f} t", "Gas (Scope 1) · unit unconfirmed",
                   "cubic-feet … documented m³ — either way the biggest source", "bad")
            + '</div><div class="gl-sec">Verified avoided emissions (counted at the low end only)</div>' + tbl(av)
            + '<div class="gl-sec">Inventory lines</div>'
            + tbl(CARBON.assign(**{"t CO2e": CARBON["t CO2e"].round(2), "amount": CARBON.amount.round(0)}).fillna("—"))
            + '<div class="gl-sec">Emission factors and sources</div>' + tbl(CARBON_FACTORS))


# ---------------- WHAT-IF ----------------
def ui_what_if(heat_down, cool_up, offday_cut, base_cut, tariff, gas_doc):
    W, kwh, m3, co2_e, co2_g = what_if(heat_down, cool_up, offday_cut, base_cut)
    if co2_g is None and gas_doc:
        co2_g = m3 * GAS_KG_CO2_PER_M3 / 1000
    t = _num(tariff)
    cards = '<div class="gl-cards">' + "".join([
        card(f"{kwh / 1000:,.0f} MWh", "⚡ Electricity / year", f"= {co2_e:,.1f} t CO₂ on Québec hydro · {kwh * INDIA_GRID_KG_PER_KWH / 1000:,.0f} t on India's grid", "info"),
        card(f"{m3:,.0f}", "🔥 Gas / year (meter units)", (f"= {co2_g:,.0f} t CO₂ if the documented m³ is right" if co2_g is not None
                                                          else "CO₂ shown once the unit is confirmed (tick the box to see it at m³)"), "warn"),
        card(f"{kwh * t:,.0f}" if t else "—", "💰 Electricity cost / year", "your tariff × kWh" if t else "type your tariff per kWh", "mute"),
    ]) + "</div>"
    f = go.Figure()
    for k, col in [("heating", "#F97316"), ("cooling", "#22D3EE"), ("weekends", "#A78BFA"), ("base load", "#10B981")]:
        f.add_trace(go.Bar(y=W.meter + " (" + W.unit + ")", x=W[k], orientation="h", name=k, marker_color=col))
    f.update_layout(barmode="stack")
    f = fig_(f, "Where the projected saving comes from (per year, last year's real weather)", 300)
    return cards + '<div class="gl-note">Projection from each building\'s own model — not a saving until it is done and verified in “Act & Prove”.</div>', f


def ui_carbon_chart(gas_doc=False):
    C = carbon_by_period(gas_doc).fillna(0)
    colors = {"Scope 2 · electricity": "#10B981", "Scope 3 · waste to landfill": "#94A3B8", "Scope 1 · natural gas": "#F97316",
              "Scope 3 · e-waste": "#A78BFA"}
    f = go.Figure([go.Bar(x=C.index, y=C[c], name=c, marker_color=colors.get(c, "#888")) for c in C.columns])
    f.update_layout(barmode="stack")
    note = "gas at the DOCUMENTED unit (m³, unconfirmed)" if gas_doc else "gas not included until its meter unit is confirmed"
    f = fig_(f, f"Measured CO₂e per 12-month period ({PERIOD_LABEL}) — {note}", 420)
    f.update_yaxes(title="t CO₂e", type="log" if gas_doc else "linear")
    return f


# ---------------- LAYOUT ----------------
HEADER = f"""{CSS}
<div class="gl-head"><div class="gl-brand"><div class="gl-logo">G</div><div>
<div class="gl-title">GreenLedger</div>
<div class="gl-sub">CAMPUS RESOURCE OPTIMISATION · ELECTRICITY · GAS · WATER · WASTE · E-WASTE · CARBON · IC26-05-01</div></div></div>
<div class="gl-pill">REFERENCE CAMPUS: CONCORDIA UNIVERSITY (MONTRÉAL) · REAL OPEN DATA · NOTHING SIMULATED</div></div>
<div class="gl-steps">
<div class="gl-step"><span class="n">1</span><b>Measure</b><p>Every meter, invoice and a live camera. Bad readings are found and excluded with a reason — never filled in.</p></div>
<div class="gl-step"><span class="n">2</span><b>Optimise</b><p>What-ifs and recommendations sized from each building's own data; live AC/fan/light control from who is in the room.</p></div>
<div class="gl-step"><span class="n">3</span><b>Prove</b><p>A ghost twin shows what would have happened without the action; placebo tests show what noise looks like. We count the low end.</p></div>
</div>
<div class="gl-intro"><b>Start the demo here →</b> ① <b>Overview</b>: slide the what-if · ② <b>Act &amp; Prove</b>: trust test, approve a recommendation, ghost twin + placebo · ③ <b>Live room</b> · ④ <b>Data</b> for the detail.</div>"""

_first_action = ACTIONS.action_id.iloc[1] if len(ACTIONS) > 1 else (ACTIONS.action_id.iloc[0] if len(ACTIONS) else None)
_v0 = ui_verify(_first_action, 42)
_ew0 = ewaste_view(EWASTE)
_wi0 = ui_what_if(1, 1, 10, 1, "", False)
if WEATHER["trm"] is None:
    fetch_weather("Bengaluru")

with gr.Blocks(title="GreenLedger") as demo:
    gr.HTML(HEADER)
    with gr.Tabs():
        # ======================= 1 · OVERVIEW =======================
        with gr.Tab("📊 Overview"):
            month_dd = gr.Dropdown(MONTH_CHOICES, value=MONTH_CHOICES[-1][1], label="Month")
            ov_cards = gr.HTML(ui_overview(MONTH_CHOICES[-1][1]))
            gr.HTML('<div class="gl-sec">🎚️ What if we… (projection from each building’s own model and last year’s real weather)</div>')
            with gr.Row():
                s_heat = gr.Slider(0, 3, value=1, step=0.5, label="Lower heating setpoint by (°C)")
                s_cool = gr.Slider(0, 3, value=1, step=0.5, label="Raise cooling setpoint by (°C)")
                s_off = gr.Slider(0, 50, value=10, step=5, label="Use less on weekends & holidays (%)")
                s_base = gr.Slider(0, 20, value=1, step=1, label="Cut always-on electricity (%)")
            with gr.Row():
                s_tariff = gr.Textbox(label="Your electricity tariff per kWh (optional)", placeholder="e.g. from the bill", scale=2)
                s_gas = gr.Checkbox(value=False, label="Show gas CO₂ at the documented unit (m³, unconfirmed)", scale=2)
            wi_html = gr.HTML(_wi0[0]); wi_plot = gr.Plot(_wi0[1])
            gr.HTML('<div class="gl-sec">Top recommendations (still waiting for a decision)</div>')
            top_html = gr.HTML(rec_cards(top_recs()))
            meter_dd = gr.Dropdown(METER_CHOICES, value="Point_7", label="Meter")
            ov_plot = gr.Plot(ui_meter_chart("Point_7"))
            gr.Plot(ui_map())
            with gr.Row():
                rep_btn = gr.Button("Build fresh report", variant="primary", scale=1)
                rep_files = gr.File(value=[REPORT_HTML, REPORT_ZIP], label="Report (HTML) + labelled dataset (ZIP)", file_count="multiple", scale=3)
        # ======================= 2 · ACT & PROVE =======================
        with gr.Tab("✅ Act & Prove"):
            gr.HTML('<div class="gl-intro"><b>Before you trust any of this</b> — check whether we are honest on months where nothing happened.</div>')
            gr.Plot(ui_trust_chart())
            gr.HTML('<div class="gl-sec">💡 Recommendations — a person approves or rejects each one</div>')
            rec_all = gr.HTML(rec_cards(RECS))
            with gr.Row():
                rec_dd = gr.Dropdown(REC_CHOICES, value=REC_CHOICES[0][1] if REC_CHOICES else None, label="Recommendation", scale=3)
                rec_by = gr.Textbox(label="Your name / role", scale=1)
                rec_date = gr.Textbox(value=datetime.date.today().isoformat(), label="Date it was done (YYYY-MM-DD)", scale=1)
            rec_reason = gr.Textbox(label="Reason (required to reject)")
            with gr.Row():
                rec_ok = gr.Button("✅ Approve", variant="primary"); rec_no = gr.Button("✋ Reject")
            rec_msg = gr.HTML(ui_rec_detail(REC_CHOICES[0][1]) if REC_CHOICES else ""); rec_log = gr.HTML(decisions_html())
            gr.HTML('<div class="gl-sec">Action register — ghost twin + placebo-in-time</div>'
                    '<div class="gl-note">Ghost twin = the building without the action, fed the real weather and calendar. Placebo = the same '
                    'test on every week of the year before. VERIFIED only if the 95% range excludes zero and p ≤ 5%. '
                    'The first entry is a ground-truth test (the documented COVID-19 closure), not a sustainability saving.</div>')
            reg = gr.HTML(register_html())
            with gr.Row():
                act_dd = gr.Dropdown(action_choices(), value=_first_action, label="Action to check", scale=4)
                win = gr.Radio([("4 weeks", 28), ("6 weeks", 42), ("8 weeks", 56)], value=42, label="Measure for", scale=2)
            v_card = gr.HTML(_v0[0]); v_cards = gr.HTML(_v0[3])
            with gr.Row():
                v_plot = gr.Plot(_v0[1]); v_hist = gr.Plot(_v0[2])
            gr.HTML('<div class="gl-sec">Log your own action</div>')
            with gr.Row():
                n_meter = gr.Dropdown(METER_CHOICES, label="Meter", scale=2)
                n_date = gr.Textbox(label="Date (YYYY-MM-DD)", placeholder="2020-05-01", scale=1)
                n_title = gr.Textbox(label="What was done", scale=3)
                n_claim = gr.Textbox(label="Claimed saving % (optional)", scale=1)
            n_btn = gr.Button("Log action", variant="primary"); n_msg = gr.HTML()
            with gr.Accordion("How the models were chosen, their accuracy, bad data and every setting", open=False):
                gr.HTML('<div class="gl-sec">Which model is each ghost twin? (tested on 12 hidden months)</div>' + tbl(MODEL_CHECK.drop(columns="point"))
                        + '<div class="gl-sec">Model accuracy (ASHRAE Guideline 14) and what it learned</div>' + tbl(ACC.drop(columns="point"))
                        + '<div class="gl-sec">Bad data we found (excluded, with reason)</div>'
                        + tbl(fault_periods(FAULTS[FAULTS.rule != "incomplete day"]))
                        + f'<div class="gl-note">Plus {int((FAULTS.rule == "incomplete day").sum())} days with fewer than 22 of 24 hours recorded.</div>'
                        + '<div class="gl-sec">Every setting and its source</div>' + tbl(PARAMETERS))
        # ======================= 3 · LIVE ROOM =======================
        with gr.Tab("🎥 Live room"):
            gr.HTML('<div class="gl-intro"><b>Camera counts people → ASHRAE comfort rules decide AC / fan / lights → empty 20 min = all off.</b></div>')
            with gr.Row():
                cam = gr.Image(sources=["webcam"], streaming=True, type="numpy", label="Live camera (allow access)", scale=1)
                photo = gr.Image(sources=["upload"], type="numpy", label="…or test with a photo of the room", scale=1)
                cam_out = gr.Image(label="What the AI sees", scale=1)
            room_html = gr.HTML(room_status_html(0, "Start the camera."))
            with gr.Row():
                r_city = gr.Textbox(value=WEATHER["city"].split(",")[0] if WEATHER["city"] else "Bengaluru", label="City (live weather)", scale=2)
                r_wbtn = gr.Button("↻ Weather", scale=1)
                r_temp = gr.Textbox(label="Room temperature °C (thermometer / AC display)", placeholder="e.g. 29", scale=2)
                r_out = gr.Textbox(label="Outdoor 7-day mean °C (only if offline)", placeholder="auto from Open-Meteo", scale=2)
            with gr.Row():
                r_auto = gr.Checkbox(value=False, label="Auto-control (switch devices)", scale=1)
                r_vac = gr.Number(value=VACANCY_MIN_DEFAULT, label="Switch off after empty (min)", scale=1)
                r_kac = gr.Textbox(label="AC nameplate kW", placeholder="from the label", scale=1)
                r_kfan = gr.Textbox(label="Fan nameplate kW", placeholder="from the label", scale=1)
                r_kl = gr.Textbox(label="Lights total kW", placeholder="sum of fittings", scale=1)
            r_est = gr.HTML()
            with gr.Row():
                b_acon = gr.Button("❄️ AC on"); b_acoff = gr.Button("❄️ AC off"); b_fon = gr.Button("🌀 Fan on")
                b_foff = gr.Button("🌀 Fan off"); b_lon = gr.Button("💡 Lights on"); b_loff = gr.Button("💡 Lights off")
            room_log = gr.Dataframe(ui_room_log(), label="Control log", interactive=False)
        # ======================= 4 · DATA =======================
        with gr.Tab("📂 Data"):
            with gr.Tabs():
                with gr.Tab("🚨 Alerts"):
                    gr.HTML('<div class="gl-intro">An alert means a meter used more than its ghost twin — after allowing for weather and day — '
                            'by more than it ever did in the whole baseline year.</div>')
                    gr.HTML(ui_alerts())
                    al_dd = gr.Dropdown([(f"{r.id} · {r.meter}", r.id) for _, r in ALERT_TABLE.iterrows()],
                                        value=ALERT_TABLE.id.iloc[0] if len(ALERT_TABLE) else None, label="Show an alert")
                    al_plot = gr.Plot(ui_alert_chart(ALERT_TABLE.id.iloc[0] if len(ALERT_TABLE) else None))
                with gr.Tab("🌍 Carbon"):
                    c_gas = gr.Checkbox(value=False, label="Include gas at the documented unit (m³, unconfirmed — log scale)")
                    c_plot = gr.Plot(ui_carbon_chart(False))
                    gr.HTML(ui_carbon_html())
                with gr.Tab("🗑️ Waste"):
                    gr.HTML('<div class="gl-intro">Diversion = compost + paper + mixed recycling ÷ (those + landfill). Bulky waste is shown but left out '
                            '— Concordia says its split is unknown.</div>')
                    gr.HTML(ui_waste_cards())
                    gr.Plot(ui_waste_monthly())
                    gr.Plot(ui_waste_buildings())
                    gr.HTML(tbl(WASTE_YEARLY.rename(columns=STREAM_LABEL).rename_axis(None, axis=1).reset_index()
                                .rename(columns={"period": f"12 months ({PERIOD_LABEL})"}).fillna("—")))
                with gr.Tab("💻 E-waste"):
                    gr.HTML('<div class="gl-intro"><b>Populate before use:</b> upload your department\'s real asset register. Routes follow India\'s '
                            'E-Waste (Management) Rules, 2022 (registered refurbisher / recycler only). Expected life = CPCB EPR average life for '
                            'the device\'s EEE code.</div>')
                    with gr.Row():
                        ew_file = gr.File(label="Upload ewaste_register.csv", file_types=[".csv"], scale=2)
                        gr.File(value=EWASTE_TEMPLATE, label="Template (CSV)", scale=1)
                    ew_html = gr.HTML(_ew0[0]); ew_plot = gr.Plot(_ew0[1])
    gr.HTML('<div class="gl-foot"><span>GreenLedger · IC26-05-01</span><span>Data: Concordia University open data · CC BY 4.0</span></div>')

    month_dd.change(ui_overview, month_dd, ov_cards)
    _wi_in = [s_heat, s_cool, s_off, s_base, s_tariff, s_gas]
    for c_ in _wi_in:
        c_.change(ui_what_if, _wi_in, [wi_html, wi_plot])
    meter_dd.change(ui_meter_chart, meter_dd, ov_plot)
    rep_btn.click(lambda: list(build_report()), None, rep_files)
    rec_dd.change(ui_rec_detail, rec_dd, rec_msg)
    _rec_out = [rec_msg, rec_log, act_dd, rec_all, reg, top_html]
    rec_ok.click(lambda r, b, why, d: ui_rec_decide(r, "approved", b, why, d), [rec_dd, rec_by, rec_reason, rec_date], _rec_out)
    rec_no.click(lambda r, b, why, d: ui_rec_decide(r, "rejected", b, why, d), [rec_dd, rec_by, rec_reason, rec_date], _rec_out)
    act_dd.change(ui_verify, [act_dd, win], [v_card, v_plot, v_hist, v_cards])
    win.change(ui_verify, [act_dd, win], [v_card, v_plot, v_hist, v_cards])
    n_btn.click(ui_add, [n_meter, n_date, n_title, n_claim], [n_msg, act_dd, reg])
    al_dd.change(ui_alert_chart, al_dd, al_plot)
    c_gas.change(ui_carbon_chart, c_gas, c_plot)
    _room_in = [cam, r_temp, r_auto, r_vac, r_out, r_kac, r_kfan, r_kl]
    _room_out = [cam_out, room_html, room_log, r_est]
    try:
        cam.stream(ui_room, _room_in, _room_out, stream_every=2.0)       # Gradio 5 / 6 — every 2 s keeps CPU YOLO smooth
    except TypeError:
        cam.stream(ui_room, _room_in, _room_out)                         # Gradio 4
    photo.upload(ui_room, [photo] + _room_in[1:], _room_out)
    r_wbtn.click(ui_weather, r_city, room_html)
    for b_, d_, on_ in [(b_acon, "AC", True), (b_acoff, "AC", False), (b_fon, "Fan", True), (b_foff, "Fan", False),
                        (b_lon, "Lights", True), (b_loff, "Lights", False)]:
        b_.click(lambda d=d_, o=on_: ui_manual(d, o), None, [room_html, room_log])
    ew_file.change(ui_ewaste_upload, ew_file, [ew_html, ew_plot])
print("App built — run the next cell to open it.")


## Cell 15 · Open the app
Click the **…gradio.live** link (works on a phone; allow camera access for the Live room tab). Keep this cell running during the demo.

**Demo-day safety:** run all cells *before* you go on stage (the YOLO model is saved locally in cell 13, so no download is needed later). Screen-record the Live room working beforehand as a backup. If the public link is blocked by venue Wi-Fi, the app opens inside the notebook.

In [ ]:
# Opens the app. In Colab, click the public link (…gradio.live) — it works on a phone too and lets the browser
# use its camera. If the venue Wi-Fi blocks the public link, the app still opens inside this notebook.
if os.environ.get("GL_NO_LAUNCH") != "1":
    try:
        demo.queue().launch(share=True, debug=False)
    except Exception as e:
        print("Public link failed:", e, "→ opening inside the notebook instead")
        demo.queue().launch(share=False, inline=True)
